# Temporal Character Networks — Analysis
## End-to-End Analytical Pipeline

**Paper**: *Temporal Dynamics of Social Balance in Fictional Character Networks*  
**Data**: 626 books, 75,482 character-relationship observations (SparkNotes corpus)

---

### Notebook Structure
| Block | Analysis 
|-------|----------
| 0 | Setup, paths, global config, colour scheme 
| 1 | Data loading, preprocessing, metadata join 
| A | Aggregate static network properties 
| B | Temporal network statistics 
| C | Structural balance theory (B(t)) 
| D1 | Character introduction rate 
| D2 | Edge polarity shift dynamics 
| D3 | Network densification 
| D4 | Protagonist centrality trajectory 
| D5 | Negative edge concentration (Gini) 
| D6 | Social entropy trajectory 
| D7 | Community formation (modularity) 
| E | Genre classification 
| F | Epoch & author comparison 
| G | Per-book outputs

---
### Key Methodological Decisions
1. **Cumulative G(t)**: edges union up to chapter t; most-recent polarity for conflicting observations  
2. **Neutral edges**: included in all density/degree metrics; **excluded** from balance triadic census  
3. **Balance threshold**: B(t) computed only where ≥5 signed (non-neutral) triads exist  
4. **Narrative time**: τ = chapter/max_chapter ∈ (0,1]  
5. **Form confound**: Tragedy/Drama = 78/81 plays; all results stratified by Form as robustness check  
6. **Data is undirected**: confirmed no (A,B) + (B,A) reciprocal pairs exist  

---
## Block 0 — Setup, Paths, Global Configuration

In [ ]:
# ============================================================
# BLOCK 0: Setup — Run this cell first, always
# ============================================================

import os
import sys
import json
import logging
import warnings
import time
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import networkx as nx

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (classification_report, confusion_matrix, f1_score, ConfusionMatrixDisplay)
from sklearn.manifold import TSNE
from sklearn.impute import SimpleImputer

warnings.filterwarnings('ignore')

# Adjust BASE_DIR to your local project root
BASE_DIR = Path('.')
ANALYSIS_DIR = BASE_DIR / 'analysis'

# Input data
NETWORK_CSV  = BASE_DIR / 'temporal_literary_networks.csv'
METADATA_CSV = BASE_DIR / 'temporal_literary_networks_metadata.csv'

# Output directories
DIRS = {
    'data':      ANALYSIS_DIR / 'data',
    'figs_A':    ANALYSIS_DIR / 'figures' / 'A_aggregate',
    'figs_B':    ANALYSIS_DIR / 'figures' / 'B_temporal',
    'figs_C':    ANALYSIS_DIR / 'figures' / 'C_balance',
    'figs_D':    ANALYSIS_DIR / 'figures' / 'D_motifs',
    'figs_E':    ANALYSIS_DIR / 'figures' / 'E_classification',
    'figs_F':    ANALYSIS_DIR / 'figures' / 'F_epoch_author',
    'figs_H':    ANALYSIS_DIR / 'figures' / 'H_centrality_polarization',
    'tables':    ANALYSIS_DIR / 'results' / 'tables',
    'stats':     ANALYSIS_DIR / 'results' / 'stats',
    'per_book':  ANALYSIS_DIR / 'per_book',
    'logs':      ANALYSIS_DIR / 'logs',
}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

# ─── LOGGING ──────────────────────────────────────────────────────
log_path = DIRS['logs'] / f"analysis_{time.strftime('%Y%m%d_%H%M%S')}.log"
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    handlers=[logging.FileHandler(log_path), logging.StreamHandler(sys.stdout)]
)
log = logging.getLogger('templitnet')
log.info(f'Analysis session started. Outputs in: {ANALYSIS_DIR}')

# GLOBAL COLOUR SCHEME
# 8 genres — semantically ordered (warm=conflict-heavy, cool=resolution-heavy)
GENRE_ORDER = ['Tragedy / Drama', 'Mystery / Thriller', 'Science Fiction / Dystopia', 'Comedy / Satire', 'Fantasy / Adventure', 'Literary / Realist Fiction', 'Coming-of-Age / Bildungsroman', 'Romance / Domestic Fiction']
GENRE_COLORS = {
    'Tragedy / Drama':               '#990000',  # deep crimson
    'Mystery / Thriller':            '#7B3F00',  # dark brown
    'Science Fiction / Dystopia':    '#005B5B',  # dark teal
    'Comedy / Satire':               '#2E7D32',  # dark green
    'Fantasy / Adventure':           '#4527A0',  # deep purple
    'Literary / Realist Fiction':    '#1565C0',  # strong blue
    'Coming-of-Age / Bildungsroman': '#E65100',  # deep orange
    'Romance / Domestic Fiction':    '#E91E8C',  # vivid hot pink
}

# Epochs — 6 periods, blue sequential palette
EPOCH_ORDER = ['Pre-Modern (pre-1600)', 'Early Modern (1600–1800)', '19th Century', 'Early 20th Century (1900–1950)', 'Late 20th Century (1950–2000)', '21st Century (2000+)']
EPOCH_COLORS = {
    'Pre-Modern (pre-1600)':          '#7B3F00',  # dark brown
    'Early Modern (1600–1800)':       '#E65100',  # burnt orange
    '19th Century':                   '#F9A825',  # amber
    'Early 20th Century (1900–1950)': '#2E7D32',  # forest green
    'Late 20th Century (1950–2000)':  '#1565C0',  # strong blue
    '21st Century (2000+)':           '#6A1B9A',  # deep purple
}

# Edge polarity colours (used consistently everywhere)
POLARITY_COLORS = {'positive': '#2ca02c', 'neutral': '#aec7e8', 'negative': '#d62728'}

# Matplotlib global style 
plt.rcParams.update({
    'font.family': 'Helvetica',
    'font.size': 12,
    'axes.titlesize': 16,
    'axes.labelsize': 14,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'figure.facecolor': 'white',
    'axes.grid': False,          # disable grid globally
})
sns.set_style('white')  # no background grid

# # Remove top and right axes spines globally
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.spines.bottom'] = False
plt.rcParams['axes.spines.left']   = False

# Global constants 
N_BINS          = 20      # bins for narrative time normalisation
MIN_TRIADS      = 5       # min signed triads for B(t) computation
N_BOOTSTRAP     = 500     # bootstrap resamples for CIs
MIN_CHARS_SW    = 10      # min characters for small-world test
N_SW_RANDOM     = 100     # random graphs for small-world benchmark
RANDOM_STATE    = 42
N_CV_FOLDS      = 10      # cross-validation folds

# Short genre labels for figure axes (full names in captions/text)
GENRE_LABELS_SHORT = {
    'Tragedy / Drama':               'Drama',
    'Mystery / Thriller':            'Mystery',
    'Science Fiction / Dystopia':    'Sci-Fi',
    'Comedy / Satire':               'Comedy',
    'Fantasy / Adventure':           'Fantasy',
    'Literary / Realist Fiction':    'Realist Fic.', 
    'Coming-of-Age / Bildungsroman': 'Coming-of-Age',
    'Romance / Domestic Fiction':    'Romance',
}

# Consistent poliarity colours (used in D2b, G2, motif plots)
POLARITY_COLORS = {
    'positive': '#2ca02c',   # green
    'negative': '#d62728',   # red
    'neutral':  '#aec7e8',   # light blue
}

# Label helper
# Use $\tau$ style mathtext instead of the UTF-8 τ character. Replace all label strings that use raw τ/⁺/⁻ with these safe versions.
TAU = r'$\tau$'       # normalised narrative time axis label
DPOS_LABEL  = r'$D^+(\tau)$'
DNEG_LABEL  = r'$D^-(\tau)$'
RPLUS_LABEL = r'$R^+(\tau)$'
HDEG_LABEL  = r'$H(\tau)$'
CAVG_LABEL  = r'$C(\tau)$'
B_LABEL     = r'$B(\tau)$'

def savefig(fig, directory_key, filename):
    """Save figure as PNG and PDF to the appropriate output directory."""
    path = DIRS[directory_key] / filename
    fig.savefig(path, dpi=300, bbox_inches='tight', facecolor='white')
    # Also save as PDF (vector quality for paper submission)
    pdf_path = path.with_suffix('.pdf')
    fig.savefig(pdf_path, bbox_inches='tight', facecolor='white')
    log.info(f'Saved figure: {path}')
    plt.show()

print(' Block 0 complete — directories created, colour scheme loaded')
print(f'   Output root: {ANALYSIS_DIR}')

---
## Block 1 — Data Loading & Preprocessing

In [ ]:
# BLOCK 1a: Load raw data & metadata join

log.info('Loading network data...')
df_raw = pd.read_csv(NETWORK_CSV, sep=';')
log.info(f'  Loaded {len(df_raw):,} rows, {df_raw["Book"].nunique()} unique books')

log.info('Loading metadata...')
meta = pd.read_csv(METADATA_CSV)

# Manual lookup for the 4 books with name mismatches
BOOK_KEY_FIXES = {
    'OE Roelvaag: Giants in the Earth':                       'OE Rlvaag: Giants in the Earth',
    'Christopher Collier James Lincoln Collier: My Brother Sam Is Dead': 'Christopher and James Lincoln Collier: My Brother Sam Is Dead',
    'Taylor Jenkins Reid: Daisy Jones the Six':               'Taylor Jenkins Reid: Daisy Jones and the Six',
    'Jerome Lawrence Robert E Lee: Inherit the Wind':         'Jerome Lawrence and Robert E Lee: Inherit the Wind',
}
meta['key'] = (meta['Author'] + ': ' + meta['Book']).replace(BOOK_KEY_FIXES)

# Join metadata onto network data
df = df_raw.merge(meta[['key', 'Author', 'Year', 'Epoch', 'Genre_Primary', 'Genre_Secondary', 'Form', 'Remarks']], left_on='Book', right_on='key', how='left').drop(columns='key')

# Validate join
missing = df['Genre_Primary'].isna().sum()
log.info(f'  Metadata join: {len(df_raw) - missing:,}/{len(df_raw):,} rows have genre ({missing} missing)')
assert missing < 500, f'Too many missing genre assignments: {missing}'

# Normalize polarity values
VALID_POLARITIES = {'positive', 'negative', 'neutral'}
unexpected = df[~df['Relationship'].isin(VALID_POLARITIES)]
if len(unexpected) > 0:
    log.warning(f'Unexpected polarity values: {unexpected["Relationship"].value_counts().to_dict()}')
    df = df[df['Relationship'].isin(VALID_POLARITIES)].copy()

# Add canonical undirected pair column 
df['pair'] = df.apply(lambda r: tuple(sorted([r['Character A'], r['Character B']])), axis=1)

# Confirm: data is undirected (sanity check)
directed_check = df.groupby(['Book', 'Chapter', 'pair']).size()
assert directed_check.max() == 1, 'Duplicate pairs found within same chapter — check data'
log.info('   Data confirmed undirected (no duplicate pairs per chapter)')

print(f' Block 1a complete')
print(f'   Books: {df["Book"].nunique()} | Edges: {len(df):,} | Genres: {df["Genre_Primary"].nunique()}')
df.head(3)

In [ ]:
# BLOCK 1b: Build per-book book registry + corpus overview

def get_all_chars(book_df):
    return set(book_df['Character A']) | set(book_df['Character B'])

# Book-level summary
book_registry = []
for book, bdf in df.groupby('Book'):
    chars = get_all_chars(bdf)
    chapters = sorted(bdf['Chapter'].unique())
    pol = bdf['Relationship'].value_counts(normalize=True)
    book_registry.append({
        'Book':         book,
        'Author':       bdf['Author'].iloc[0],
        'Year':         bdf['Year'].iloc[0] if 'Year' in bdf else None,
        'Epoch':        bdf['Epoch'].iloc[0],
        'Genre':        bdf['Genre_Primary'].iloc[0],
        'Form':         bdf['Form'].iloc[0],
        'n_chars':      len(chars),
        'n_edges_raw':  len(bdf),
        'n_unique_pairs': bdf['pair'].nunique(),
        'n_chapters':   len(chapters),
        'max_chapter':  max(chapters),
        'frac_pos':     pol.get('positive', 0),
        'frac_neg':     pol.get('negative', 0),
        'frac_neu':     pol.get('neutral', 0),
    })

registry = pd.DataFrame(book_registry)
registry.to_csv(DIRS['data'] / 'book_registry.csv', index=False)

# Quick corpus overview 
print('=== CORPUS OVERVIEW ===')
print(f'Total books: {len(registry)}')
print(f'Total edges: {registry["n_edges_raw"].sum():,}')
print(f'Authors: {registry["Author"].nunique()}')
print(f'Chapters range: {registry["n_chapters"].min()}–{registry["n_chapters"].max()} (median {registry["n_chapters"].median():.0f})')
print(f'Characters range: {registry["n_chars"].min()}–{registry["n_chars"].max()} (median {registry["n_chars"].median():.1f})')
print('\nGenre distribution:')
print(registry['Genre'].value_counts().to_string())
print('\nForm distribution:')
print(registry['Form'].value_counts().to_string())
print('\n⚠️  Form × Genre confound:')
print(registry.groupby(['Genre', 'Form']).size().unstack(fill_value=0))

print('\n Block 1b complete')

In [ ]:
# BLOCK 1c: Build cumulative network builder (shared utility)

# Function used by ALL downstream temporal analyses. It implements the core methodological decision:
#   - Cumulative G(t): union of all edges seen in chapters 1..t
#   - Polarity: most-recent-wins when same pair seen in multiple chapters
# Returns a sequence of networkx graphs, one per chapter.

def build_cumulative_networks(book_df, return_graphs=True):
    chapters = sorted(book_df['Chapter'].unique())
    edge_dict = {}  # pair -> latest polarity
    edge_dicts = []
    graphs = [] if return_graphs else None

    for ch in chapters:
        ch_data = book_df[book_df['Chapter'] == ch]
        for _, row in ch_data.iterrows():
            key = tuple(sorted([row['Character A'], row['Character B']]))
            edge_dict[key] = row['Relationship']  # most recent wins

        edge_dicts.append(dict(edge_dict))  # snapshot at this t

        if return_graphs:
            G = nx.Graph()
            for (a, b), pol in edge_dict.items():
                G.add_edge(a, b, polarity=pol, sign=(
                    1 if pol == 'positive' else (-1 if pol == 'negative' else 0)
                ))
            graphs.append(G)

    return chapters, edge_dicts, graphs

# Count closed signed triads. Exclude triads w any neutral edge. Return dict w counts of each triad type.
def classify_triads(edge_dict):
    # Only +/- edges
    signed_edges = {k: v for k, v in edge_dict.items() if v != 'neutral'}
    nodes = set()
    for a, b in signed_edges.keys():
        nodes.update([a, b])
    nodes = list(nodes)

    counts = {'ppp': 0, 'ppn': 0, 'pnn': 0, 'nnn': 0, 'total': 0}

    for a, b, c in combinations(nodes, 3):
        ab = tuple(sorted([a, b]))
        bc = tuple(sorted([b, c]))
        ac = tuple(sorted([a, c]))
        if ab in signed_edges and bc in signed_edges and ac in signed_edges:
            n_neg = sum(1 for e in [ab, bc, ac] if signed_edges[e] == 'negative')
            counts['total'] += 1
            if   n_neg == 0: counts['ppp'] += 1
            elif n_neg == 1: counts['ppn'] += 1
            elif n_neg == 2: counts['pnn'] += 1
            else:            counts['nnn'] += 1

    return counts

print(' Block 1c complete — cumulative network builder defined')

---
## Block A — Aggregate Static Network Properties

In [ ]:
# BLOCK A1: Compute static metrics for every book

# Uses the FULL cumulative network (all chapters collapsed). Small-world sigma: (C/C_rand) / (L/L_rand), averaged over N_SW_RANDOM random graphs.

# Compute all static network metrics for one book (full cumulative).
def compute_static_metrics(book, bdf):
    # Build full cumulative network
    _, edge_dicts, graphs = build_cumulative_networks(bdf, return_graphs=True)
    G = graphs[-1]  # final state = full book

    N = G.number_of_nodes()
    E = G.number_of_edges()
    if N < 2:
        return None

    density = nx.density(G)
    avg_degree = 2 * E / N
    degrees = [d for _, d in G.degree()]
    degree_skew = float(stats.skew(degrees)) if len(degrees) > 2 else np.nan

    C_global = nx.transitivity(G)
    C_avg    = nx.average_clustering(G)
    n_comp   = nx.number_connected_components(G)
    lcc      = G.subgraph(max(nx.connected_components(G), key=len)).copy()
    f_lcc    = lcc.number_of_nodes() / N

    # Average path length on LCC
    if lcc.number_of_nodes() > 2:
        L = nx.average_shortest_path_length(lcc)
    else:
        L = np.nan

    # Assortativity
    try:
        assort = nx.degree_assortativity_coefficient(G)
    except Exception:
        assort = np.nan

    # Edge polarity fractions
    all_pols = bdf['Relationship'].value_counts(normalize=True)
    f_pos = all_pols.get('positive', 0)
    f_neg = all_pols.get('negative', 0)
    f_neu = all_pols.get('neutral', 0)

    # ── Small-world sigma (only for books with >= MIN_CHARS_SW nodes) ──
    sigma_sw = np.nan
    omega_sw = np.nan
    if N >= MIN_CHARS_SW and not np.isnan(L):
        rand_C_list, rand_L_list = [], []
        for _ in range(N_SW_RANDOM):
            R = nx.gnm_random_graph(N, E, seed=None)
            rand_C_list.append(nx.average_clustering(R))
            if nx.is_connected(R) and R.number_of_nodes() > 2:
                rand_L_list.append(nx.average_shortest_path_length(R))
        C_rand = np.mean(rand_C_list) if rand_C_list else np.nan
        L_rand = np.mean(rand_L_list) if rand_L_list else np.nan
        if C_rand > 0 and L_rand > 0:
            sigma_sw = (C_avg / C_rand) / (L / L_rand)
            # omega: (L_rand/L) - (C_avg/C_latt); skip C_latt (expensive), report sigma only

    return {
        'Book': book,
        'N': N, 'E': E, 'density': density, 'avg_degree': avg_degree,
        'degree_skew': degree_skew,
        'C_global': C_global, 'C_avg': C_avg,
        'n_components': n_comp, 'f_lcc': f_lcc,
        'avg_path_length': L, 'assortativity': assort,
        'f_pos': f_pos, 'f_neg': f_neg, 'f_neu': f_neu,
        'sigma_sw': sigma_sw,
    }

log.info('Computing static metrics for all books...')
t0 = time.time()
static_results = []
books_list = df['Book'].unique()

for i, book in enumerate(books_list):
    bdf = df[df['Book'] == book]
    result = compute_static_metrics(book, bdf)
    if result:
        static_results.append(result)
    if (i + 1) % 50 == 0:
        log.info(f'  Processed {i+1}/{len(books_list)} books ({time.time()-t0:.0f}s)')

df_static = pd.DataFrame(static_results)
df_static = df_static.merge(registry[['Book', 'Genre', 'Epoch', 'Form', 'Author', 'Year']], on='Book', how='left')
df_static.to_csv(DIRS['tables'] / 'A1_network_stats_per_book.csv', index=False)

print(f' Block A1 complete in {time.time()-t0:.0f}s — {len(df_static)} books processed')
print(f'\n--- Static metrics summary ---')
print(df_static[['N','E','density','C_avg','avg_path_length','sigma_sw']].describe().round(3))

In [ ]:
# BLOCK A2: Table 1 — Per-genre summary statistics

METRICS_TABLE = ['N', 'E', 'density', 'avg_degree', 'C_avg', 'avg_path_length',
                 'sigma_sw', 'f_pos', 'f_neg', 'assortativity']

table1 = df_static.groupby('Genre')[METRICS_TABLE].agg(['mean', 'std']).round(3)
table1.columns = ['_'.join(c) for c in table1.columns]
table1 = table1.reindex(GENRE_ORDER)
table1.to_csv(DIRS['tables'] / 'A2_corpus_summary_table1.csv')

print('=== TABLE 1 — Corpus Statistics by Genre ===')
display(table1[[
    'N_mean','N_std','density_mean','density_std',
    'C_avg_mean','sigma_sw_mean','f_pos_mean','f_neg_mean'
]].round(3))

In [ ]:
# BLOCK A3: Figure — Small-world sigma distribution

sw_data = df_static.dropna(subset=['sigma_sw'])
print(f'Books with sigma_sw computed: {len(sw_data)} / {len(df_static)}')
print(f'Small-world (sigma > 1): {(sw_data["sigma_sw"] > 1).sum()} ({100*(sw_data["sigma_sw"] > 1).mean():.1f}%)')
print(f'Median sigma: {sw_data["sigma_sw"].median():.2f}')

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: histogram of sigma overall
ax = axes[0]
ax.hist(sw_data['sigma_sw'], bins=30, color='#369acc', edgecolor='white', linewidth=0.5)
ax.axvline(1.0, color='#d62728', linestyle='--', linewidth=1.8, label='σ = 1 (threshold)')
ax.axvline(sw_data['sigma_sw'].median(), color='#2ca02c', linestyle='-', linewidth=1.8, label=f'Median σ = {sw_data["sigma_sw"].median():.2f}')
ax.set_xlabel('Small-world coefficient σ')
ax.set_ylabel('Number of books')
ax.set_title('Small-World Coefficient Distribution\n(626 fictional character networks)')
ax.legend(frameon=False)
pct = 100 * (sw_data['sigma_sw'] > 1).mean()
ax.text(0.97, 0.95, f'{pct:.0f}% are small-world (σ > 1)', transform=ax.transAxes, ha='right', va='top', fontsize=10, bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))

# Right: boxplot by genre
ax = axes[1]
genre_order_present = [g for g in GENRE_ORDER if g in sw_data['Genre'].values]
data_by_genre = [sw_data[sw_data['Genre'] == g]['sigma_sw'].dropna().values
                 for g in genre_order_present]
# bp = ax.boxplot(data_by_genre, labels=[g.replace(' / ',' /\n') for g in genre_order_present],
bp = ax.boxplot(data_by_genre, labels=[GENRE_LABELS_SHORT.get(g, g) for g in genre_order_present], patch_artist=True, notch=False, medianprops=dict(color='black', linewidth=1.5), flierprops=dict(marker='o', markersize=3, alpha=0.4))
for patch, genre in zip(bp['boxes'], genre_order_present):
    patch.set_facecolor(GENRE_COLORS[genre])
    patch.set_alpha(0.85)
ax.axhline(1.0, color='#d62728', linestyle='--', linewidth=1.5, label='σ = 1')
ax.set_xlabel('Genre')
ax.set_xlabel('Small-world coefficient (σ, higher = more small-world)')
ax.set_title('Fictional Character Networks Are Universally Small-World')
ax.tick_params(axis='x', labelsize=10)
ax.legend(frameon=False)

fig.suptitle('Fictional Character Networks Exhibit Small-World Properties', fontsize=18, y=1.01, fontweight='bold')
plt.tight_layout()
savefig(fig, 'figs_A', 'A2_smallworld_sigma_distribution.png')
savefig(fig, 'figs_A', 'A2_smallworld_sigma_distribution.pdf')

In [ ]:
# BLOCK A4: Figure — Edge polarity by genre (stacked bar)

# Per-genre shading: 3 shades of each genre's own colour. Negative = darkest shade, positive = medium, neutral = lightest.
# Genres sorted by negative fraction (decreasing) for visual clarity.

import colorsys
import matplotlib.colors as mc

# Adjust ligthness of hex_color. lf < 1 -> darker; > 1 -> lighter
def shade_color(hex_color, lf):
    r, g, b = mc.to_rgb(hex_color)
    h, l, s = colorsys.rgb_to_hls(r, g, b)
    return colorsys.hls_to_rgb(h, min(0.92, max(0.08, l * lf)), s)

pol_by_genre = df_static.groupby('Genre')[['f_pos', 'f_neg', 'f_neu']].mean()
pol_by_genre = pol_by_genre.reindex(GENRE_ORDER).dropna()
pol_sorted = pol_by_genre.sort_values('f_neg', ascending=False)  # high-conflict first

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(pol_sorted))
width = 0.6

for i, (genre, row) in enumerate(pol_sorted.iterrows()):
    bc = GENRE_COLORS.get(genre, '#555555')
    col_pos = shade_color(bc, 0.95)   # medium shade  -> positive
    col_neu = shade_color(bc, 1.65)   # lightest shade -> neutral
    col_neg = shade_color(bc, 0.4)   # darkest shade  -> negative

    ax.bar(i, row['f_pos'], width, color=col_pos, alpha=0.75)
    ax.bar(i, row['f_neu'], width, bottom=row['f_pos'],              color=col_neu, alpha=0.75)
    ax.bar(i, row['f_neg'], width, bottom=row['f_pos']+row['f_neu'], color=col_neg, alpha=0.75)

    # Annotate ALL three segments with their percentage. Positive (bottom segment) — white text
    if row['f_pos'] > 0.06:  # only label if segment is tall enough to fit text
        ax.text(i, row['f_pos'] / 2, f"{row['f_pos']:.0%}", ha='center', va='center', fontsize=10, color='white', fontweight='bold')
    # Neutral (middle segment) — dark text (readable on light shade)
    if row['f_neu'] > 0.06:
        ax.text(i, row['f_pos'] + row['f_neu'] / 2, f"{row['f_neu']:.0%}", ha='center', va='center', fontsize=10, color='#333333', fontweight='bold')
    # Negative (top segment) — white text
    if row['f_neg'] > 0.06:
        ax.text(i, row['f_pos'] + row['f_neu'] + row['f_neg'] / 2, f"{row['f_neg']:.0%}", ha='center', va='center', fontsize=10, color='white', fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels([GENRE_LABELS_SHORT.get(g, g) for g in pol_sorted.index], fontsize=15)
ax.set_ylabel('Fraction of edges', fontsize=16)
ax.set_title('Social Tone by Genre: Edge Polarity Composition', fontweight='bold')
ax.set_ylim(0, 1)

# Schematic legend (shade meaning, not genre-specific)
legend_patches = [
    mpatches.Patch(color='#dddddd', label='Neutral (lightest shade per genre)'),
    mpatches.Patch(color='#888888', label='Positive (medium shade per genre)'),
    mpatches.Patch(color='#333333', label='Negative (darkest shade per genre)'),
]
ax.legend(handles=legend_patches, loc='upper center', bbox_to_anchor=(0.5, -0.13), ncol=3, frameon=False, fontsize=13)
plt.subplots_adjust(bottom=0.16)

plt.tight_layout()
savefig(fig, 'figs_A', 'A3_edge_polarity_by_genre.png')
savefig(fig, 'figs_A', 'A3_edge_polarity_by_genre.pdf')


In [ ]:
# BLOCK A5: Figure — Network density & size by genre (boxplots)

fig, axes = plt.subplots(1, 3, figsize=(20, 5))
metrics_labels = [('density', 'Network Density'), ('N', 'Number of Characters'), ('C_avg', 'Avg Clustering Coefficient')]

for ax, (metric, label) in zip(axes, metrics_labels):
    data = [df_static[df_static['Genre'] == g][metric].dropna().values
            for g in GENRE_ORDER if g in df_static['Genre'].values]
    genres_present = [g for g in GENRE_ORDER if g in df_static['Genre'].values]
    bp = ax.boxplot(data, labels=[GENRE_LABELS_SHORT.get(g, g) for g in genres_present], patch_artist=True, medianprops=dict(color='black', linewidth=1.5), flierprops=dict(marker='o', markersize=2, alpha=0.3))
    for patch, g in zip(bp['boxes'], genres_present):
        patch.set_facecolor(GENRE_COLORS[g])
        patch.set_alpha(0.85)
    ax.set_ylabel(label)
    ax.set_title(f'{label} by Genre')
    ax.tick_params(axis='x', labelsize=9)

fig.suptitle('Aggregate Network Properties by Genre', fontsize=18, y=1.01, fontweight='bold')
plt.tight_layout()
savefig(fig, 'figs_A', 'A4_network_properties_by_genre.png')
savefig(fig, 'figs_A', 'A4_network_properties_by_genre.pdf')

In [ ]:
# BLOCK A6: Statistical tests — Kruskal-Wallis for genre differences

from scipy.stats import kruskal

test_metrics = ['density', 'N', 'avg_degree', 'C_avg', 'f_pos', 'f_neg', 'assortativity', 'sigma_sw']

kw_results = []
for metric in test_metrics:
    groups = [df_static[df_static['Genre'] == g][metric].dropna().values
              for g in GENRE_ORDER if g in df_static['Genre'].values]
    groups = [g for g in groups if len(g) >= 3]
    if len(groups) >= 2:
        H, p = kruskal(*groups)
        # Effect size eta-squared
        N_total = sum(len(g) for g in groups)
        k = len(groups)
        eta2 = (H - k + 1) / (N_total - k)
        kw_results.append({
            'metric': metric, 'H': round(H, 3), 'p_value': round(p, 6),
            'significant': p < 0.05, 'eta_squared': round(max(0, eta2), 4)
        })

kw_df = pd.DataFrame(kw_results)
kw_df.to_csv(DIRS['stats'] / 'A_kruskal_wallis_static.csv', index=False)

print('=== Kruskal-Wallis Tests: Genre Differences in Static Metrics ===')
display(kw_df)
print(f"\n{kw_df['significant'].sum()}/{len(kw_df)} metrics show significant genre differences (p < 0.05)")

---
## Block B — Temporal Network Statistics

In [ ]:
# BLOCK B1: Compute full temporal metrics for every book

# Compute temporal network metrics for one book, at each cumulative chapter.
def compute_temporal_metrics(book, bdf):
    genre  = bdf['Genre_Primary'].iloc[0]
    form   = bdf['Form'].iloc[0]
    author = bdf['Author'].iloc[0]

    chapters, edge_dicts, graphs = build_cumulative_networks(bdf, return_graphs=True)
    K = max(chapters)  # for tau normalisation
    N_total = len(get_all_chars(bdf))  # total unique characters in book

    all_chars_seen = set()
    all_pairs_seen = set()
    rows = []

    for idx, (ch, ed, G) in enumerate(zip(chapters, edge_dicts, graphs)):
        tau = ch / K  # normalised narrative time

        # New characters this chapter
        ch_data = bdf[bdf['Chapter'] == ch]
        new_chars = (set(ch_data['Character A']) | set(ch_data['Character B'])) - all_chars_seen
        new_pairs = set(ch_data['pair']) - all_pairs_seen
        all_chars_seen.update(new_chars)
        all_pairs_seen.update(set(ch_data['pair']))

        N_t = G.number_of_nodes()
        E_t = G.number_of_edges()
        dens = nx.density(G) if N_t > 1 else 0

        # Signed densities
        n_pos = sum(1 for _, _, d in G.edges(data=True) if d.get('polarity') == 'positive')
        n_neg = sum(1 for _, _, d in G.edges(data=True) if d.get('polarity') == 'negative')
        n_neu = E_t - n_pos - n_neg
        denom_density = N_t * (N_t - 1) / 2 if N_t > 1 else 1
        d_pos = n_pos / denom_density
        d_neg = n_neg / denom_density
        r_plus = n_pos / (n_pos + n_neg) if (n_pos + n_neg) > 0 else np.nan

        # Clustering
        C_avg = nx.average_clustering(G) if N_t > 2 else np.nan

        # Degree entropy
        degrees = np.array([d for _, d in G.degree()])
        if degrees.sum() > 0:
            p = degrees / degrees.sum()
            p = p[p > 0]
            H_deg = float(-np.sum(p * np.log2(p)))
        else:
            H_deg = 0.0

        # Character intro fraction
        f_chars_intro = len(all_chars_seen) / N_total if N_total > 0 else np.nan

        rows.append({
            'Book': book, 'Genre': genre, 'Form': form, 'Author': author,
            'chapter': ch, 'tau': tau,
            'N_t': N_t, 'E_t': E_t,
            'delta_N': len(new_chars), 'delta_E': len(new_pairs),
            'density': dens, 'd_pos': d_pos, 'd_neg': d_neg,
            'f_pos': n_pos / E_t if E_t > 0 else np.nan,
            'f_neg': n_neg / E_t if E_t > 0 else np.nan,
            'r_plus': r_plus,
            'C_avg': C_avg,
            'H_deg': H_deg,
            'f_chars_intro': f_chars_intro,
        })

    return pd.DataFrame(rows)


log.info('Computing temporal metrics for all books...')
t0 = time.time()
temp_results = []

for i, book in enumerate(books_list):
    bdf = df[df['Book'] == book]
    result = compute_temporal_metrics(book, bdf)
    temp_results.append(result)
    if (i + 1) % 100 == 0:
        log.info(f'  Temporal metrics: {i+1}/{len(books_list)} ({time.time()-t0:.0f}s)')

df_temporal = pd.concat(temp_results, ignore_index=True)
df_temporal.to_parquet(DIRS['data'] / 'B_temporal_metrics.parquet', index=False)

print(f' Block B1 complete in {time.time()-t0:.0f}s')
print(f'  Rows: {len(df_temporal):,} | Books: {df_temporal["Book"].nunique()}')
df_temporal.head(3)

In [ ]:
# BLOCK B2: Genre-mean trajectory computation utility

# Interpolates each book's temporal trajectory onto N_BINS bins of tau, then computes genre mean + bootstrap 95% CI.

TAU_BINS = np.linspace(0.05, 1.0, N_BINS)  # bin centres

# Interpolate one book's temporal metric onto fixed tau bins
def interpolate_to_bins(book_df, metric, tau_bins=TAU_BINS):
    book_df = book_df.dropna(subset=['tau', metric]).sort_values('tau')
    if len(book_df) < 2:
        return np.full(len(tau_bins), np.nan)
    return np.interp(tau_bins, book_df['tau'].values, book_df[metric].values)

# Compute genre-mean trajectory + 95% bootstrap CI for a given metric. Return dict: genre -> {'mean', 'lower', 'upper', 'traces'}
def genre_mean_ci(df_temp, metric, genres=GENRE_ORDER, tau_bins=TAU_BINS,
                  n_bootstrap=N_BOOTSTRAP, rng_seed=RANDOM_STATE):
    rng = np.random.default_rng(rng_seed)
    results = {}
    for genre in genres:
        genre_books = df_temp[df_temp['Genre'] == genre]['Book'].unique()
        if len(genre_books) == 0:
            continue
        traces = np.array([
            interpolate_to_bins(df_temp[df_temp['Book'] == b], metric)
            for b in genre_books
        ])
        # Bootstrap CI
        boot_means = np.array([
            np.nanmean(traces[rng.choice(len(traces), len(traces), replace=True)], axis=0)
            for _ in range(n_bootstrap)
        ])
        results[genre] = {
            'mean':   np.nanmean(traces, axis=0),
            'lower':  np.nanpercentile(boot_means, 2.5, axis=0),
            'upper':  np.nanpercentile(boot_means, 97.5, axis=0),
            'traces': traces,
            'n_books': len(genre_books),
        }
    return results

# Produce the signatre 'fMRI-style' genre trajectory plot. Individual book traces (transparent) + genre mean (thick) + 95% CI band.
def plot_genre_trajectories(results, metric_label, title, filename, fig_dir='figs_B', tau_bins=TAU_BINS, show_individual=True, alpha_trace=0.00, ci_alpha=0.065):
    fig, ax = plt.subplots(figsize=(12, 6))
    legend_handles = []

    for genre, data in results.items():
        color = GENRE_COLORS.get(genre, 'grey')
        # Individual traces
        if show_individual:
            for trace in data['traces']:
                ax.plot(tau_bins, trace, color=color, alpha=alpha_trace, linewidth=0.7)
        # CI band
        ax.fill_between(tau_bins, data['lower'], data['upper'], color=color, alpha=ci_alpha)
        # Genre mean
        short_lbl = GENRE_LABELS_SHORT.get(genre, genre)
        line, = ax.plot(tau_bins, data['mean'], color=color, linewidth=3.0, label=f"{short_lbl} (n={data['n_books']})")
        legend_handles.append(line)

    ax.set_xlabel(r'Normalised narrative time ($\tau$)', fontsize=16)
    ax.set_ylabel(metric_label, fontsize=15)
    ax.set_title(title, fontsize=18, fontweight='bold')
    ax.set_xlim(0, 1)
    ax.legend(handles=legend_handles, frameon=False, fontsize=14, bbox_to_anchor=(1.01, 1), loc='upper left')
    plt.tight_layout()
    savefig(fig, fig_dir, filename)

# Plot genre trajectories for a selected subset of genres only. genres_subset: list of genre strings to include, eg ['Tragedy / Drama', 'Comedy / Satire']. Filename is auto-generated from genre names.
def plot_genre_subset(results, metric_label, title_prefix, filename_prefix, genres_subset, fig_dir='figs_B', tau_bins=TAU_BINS, alpha_trace=0.10):
    # Filter results to requested genres only
    subset_results = {g: v for g, v in results.items() if g in genres_subset}
    if not subset_results:
        print(f'No matching genres found in results. Check genre names.')
        return

    # Auto-generate filename suffix from genre names
    short_names = [g.split(' / ')[0].lower().replace(' ', '_').replace('-', '')
                   for g in genres_subset]
    auto_suffix = '_vs_'.join(short_names)
    filename = f'{filename_prefix}_{auto_suffix}.png'

    # Auto-generate title
    title = f'{title_prefix}\n({" vs. ".join(g.split(" / ")[0] for g in genres_subset)})'

    plot_genre_trajectories(subset_results, metric_label, title, filename, fig_dir=fig_dir, alpha_trace=alpha_trace)

print(' Block B2 complete — trajectory utility functions defined')

In [ ]:
# BLOCK B3: Figure — Network Density D(t) by genre

density_results = genre_mean_ci(df_temporal, 'density')
plot_genre_trajectories(
    density_results,
    metric_label='Fraction of possible character connections present',
    title='Network Density Over Narrative Time by Genre',
    filename='B1_density_trajectory_by_genre.png'
)

In [ ]:
# BLOCK B4: Figure — Signed density (positive / negative split)

dpos_results = genre_mean_ci(df_temporal, 'd_pos')
dneg_results = genre_mean_ci(df_temporal, 'd_neg')

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=False)

for ax, results, label, title in [
    (axes[0], dpos_results, 'Fraction of positive character connections', 'Alliances'),
    (axes[1], dneg_results, 'Fraction of negative character connections', 'Antagonisms'),
]:
    for genre, data in results.items():
        color = GENRE_COLORS.get(genre, 'grey')
        ax.fill_between(TAU_BINS, data['lower'], data['upper'], color=color, alpha=0.065)
        ax.plot(TAU_BINS, data['mean'], color=color, linewidth=2.3,
                label=f"{GENRE_LABELS_SHORT.get(genre, genre)} (n={data['n_books']})")
    ax.set_xlabel(r'Normalised narrative time ($\tau$)', fontsize=12)
    ax.set_ylabel(label, fontsize=12)
    ax.set_title(f'{title} Density by Genre')
    ax.set_xlim(0, 1)

axes[1].legend(frameon=False, fontsize=12, bbox_to_anchor=(1.01, 1), loc='upper left')
fig.suptitle('How Alliance and Antagonism Densities Evolve Across Narrative Time', fontsize=20, y=1.01, fontweight='bold')
plt.tight_layout()
savefig(fig, 'figs_B', 'B2_signed_density_by_genre.png')
savefig(fig, 'figs_B', 'B2_signed_density_by_genre.pdf')

In [ ]:
# BLOCK B5: Figures — Positive ratio R+(t), Clustering C(t), Degree entropy H(t) by genre

for metric, label, title, fname in [
    ('r_plus', r'Positive ratio $R^+(\tau)$ = pos/(pos+neg)',
     'Positive-Relationship Ratio Over Narrative Time by Genre', 'B3_positive_ratio_by_genre.png'),
    ('C_avg', r'Average clustering coefficient $C(\tau)$',
     'Clustering Coefficient Evolution by Genre', 'B4_clustering_by_genre.png'),
    ('H_deg', r'Degree distribution entropy $H(\tau)$',
     'Social Entropy (Network Egalitarianism) Over Narrative Time by Genre', 'B5_entropy_by_genre.png'),
]:
    res = genre_mean_ci(df_temporal, metric)
    plot_genre_trajectories(res, label, title, fname)

In [ ]:
# BLOCK B6: Form-stratified density (Plays vs Novels)
# ROBUSTNESS CHECK — addresses the Form confound

fig, axes = plt.subplots(1, 2, figsize=(18, 6), sharey=True)

for ax, form_val in zip(axes, ['Play', 'Novel']):
    df_form = df_temporal[df_temporal['Form'] == form_val]
    results = genre_mean_ci(df_form, 'density')

    for genre, data in results.items():
        color = GENRE_COLORS.get(genre, 'grey')
        ax.fill_between(TAU_BINS, data['lower'], data['upper'], color=color, alpha=0.065)
        ax.plot(TAU_BINS, data['mean'], color=color, linewidth=2.3, label=f"{GENRE_LABELS_SHORT.get(genre, genre)} (n={data['n_books']})")

    ax.set_xlabel('Normalised narrative time (τ)', fontsize=14)
    ax.set_ylabel('Network density D(τ)', fontsize=14)
    n = df_form['Book'].nunique()
    ax.set_title(f'{form_val}s Only – n={n} books')
    ax.set_xlim(0, 1)

axes[1].legend(frameon=False, fontsize=10, bbox_to_anchor=(1.01, 1), loc='upper left')
fig.suptitle('Density Trajectories Stratified by Form\n'
             '(Robustness check: genre patterns should hold within each form)', fontsize=18, fontweight='bold', y=1.02)
plt.tight_layout()
savefig(fig, 'figs_B', 'B6_density_form_stratified.png')
savefig(fig, 'figs_B', 'B6_density_form_stratified.pdf')

---
## Block C — Structural Balance Theory

In [ ]:
# BLOCK C1: Compute B(t) for every book at every chapter

# B(t) = fraction of closed signed triads that are balanced
# Balanced: +++ (n_neg=0) or +-- (n_neg=2)
# Imbalanced: ++- (n_neg=1) or --- (n_neg=3)
# Excludes triads containing any neutral edge
# Minimum threshold: >= MIN_TRIADS (=5) signed triads for B(t) to be valid

# Compute B(t), weak B(t), and triad fractions per chapter for one book
def compute_balance_trajectory(book, bdf):
    genre  = bdf['Genre_Primary'].iloc[0]
    form   = bdf['Form'].iloc[0]

    chapters, edge_dicts, _ = build_cumulative_networks(bdf, return_graphs=False)
    K = max(chapters)
    rows = []

    for ch, ed in zip(chapters, edge_dicts):
        tau = ch / K
        counts = classify_triads(ed)  # {ppp, ppn, pnn, nnn, total}
        total = counts['total']

        if total >= MIN_TRIADS:
            # Strong balance: balanced = ppp + pnn
            B_strong = (counts['ppp'] + counts['pnn']) / total
            # Weak balance: balanced = pnn only (enemy of enemy hypothesis)
            B_weak   = counts['pnn'] / total
            f_ppp    = counts['ppp'] / total
            f_ppn    = counts['ppn'] / total
            f_pnn    = counts['pnn'] / total
            f_nnn    = counts['nnn'] / total
        else:
            B_strong = B_weak = f_ppp = f_ppn = f_pnn = f_nnn = np.nan

        # Positive ratio (all edges, includes neutral)
        signed = {k: v for k, v in ed.items() if v != 'neutral'}
        n_pos  = sum(1 for v in signed.values() if v == 'positive')
        n_neg  = sum(1 for v in signed.values() if v == 'negative')
        r_plus = n_pos / (n_pos + n_neg) if (n_pos + n_neg) > 0 else np.nan

        rows.append({
            'Book': book, 'Genre': genre, 'Form': form,
            'chapter': ch, 'tau': tau,
            'n_triads': total,
            'B': B_strong, 'B_weak': B_weak,
            'f_ppp': f_ppp, 'f_ppn': f_ppn, 'f_pnn': f_pnn, 'f_nnn': f_nnn,
            'r_plus': r_plus,
        })

    return pd.DataFrame(rows)

log.info('Computing balance trajectories...')
t0 = time.time()
balance_results = []

for i, book in enumerate(books_list):
    bdf = df[df['Book'] == book]
    result = compute_balance_trajectory(book, bdf)
    balance_results.append(result)
    if (i + 1) % 100 == 0:
        log.info(f'  Balance: {i+1}/{len(books_list)} ({time.time()-t0:.0f}s)')

df_balance = pd.concat(balance_results, ignore_index=True)
df_balance.to_parquet(DIRS['data'] / 'C_balance_metrics.parquet', index=False)

n_valid = df_balance.dropna(subset=['B'])['Book'].nunique()
print(f' Block C1 complete in {time.time()-t0:.0f}s')
print(f'  Books with valid B(t) data: {n_valid}/{len(books_list)}')
print(f'  Books filtered (< {MIN_TRIADS} triads): {len(books_list) - n_valid}')

In [ ]:
# BLOCK C2: Figure — Balance index B(t) by genre

balance_genre_results = genre_mean_ci(df_balance, 'B')
plot_genre_trajectories(
    balance_genre_results,
    metric_label='Proportion of relationship triangles satisfying social balance',
    title='Social Balance Over Narrative Time by Genre',
    filename='C1_balance_trajectory_by_genre.png',
    fig_dir='figs_C'
)

In [ ]:
# BLOCK C3: Figure — Strong vs Weak balance comparison

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=False)

for ax, metric, label in [
    (axes[0], 'B', 'Balance index (strong criterion)\nHeider: +++ or +−− triads'),
    (axes[1], 'B_weak', 'Balance index (weak criterion)\nHarary: +−− triads only'),
]:
    res = genre_mean_ci(df_balance, metric)
    for genre, data in res.items():
        color = GENRE_COLORS.get(genre, 'grey')
        ax.fill_between(TAU_BINS, data['lower'], data['upper'], color=color, alpha=0.065)
        ax.plot(TAU_BINS, data['mean'], color=color, linewidth=2.3, label=f"{GENRE_LABELS_SHORT.get(genre, genre)} (n={data['n_books']})")
    ax.set_xlabel('Normalised narrative time (τ)')
    ax.set_ylabel(label)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

axes[1].legend(frameon=False, fontsize=8, bbox_to_anchor=(1.01, 1), loc='upper left')
fig.suptitle('Strong vs. Weak Structural Balance by Genre\n'
             '(Heider 1946 vs. Harary 1953 formulations)', fontsize=18, fontweight='bold', y=1.02)
plt.tight_layout()
savefig(fig, 'figs_C', 'C2_strong_vs_weak_balance.png')
savefig(fig, 'figs_C', 'C2_strong_vs_weak_balance.pdf')

In [ ]:
# BLOCK C4: argmin(B(t)) — When does maximum imbalance occur?

# For each book, find tau at which B(t) is minimised (= most imbalanced)
argmin_results = []
for book, bdf_b in df_balance.groupby('Book'):
    valid = bdf_b.dropna(subset=['B'])
    if len(valid) < 3:
        continue
    low_confidence = (len(valid) < 5)  # flag: argmin unreliable with < 5 valid slices
    idx_min = valid['B'].idxmin()
    argmin_tau = valid.loc[idx_min, 'tau']
    B_min  = valid.loc[idx_min, 'B']
    B_end  = valid.iloc[-1]['B']
    B_start= valid.iloc[0]['B']
    genre  = valid.iloc[0]['Genre']
    form   = valid.iloc[0]['Form']
    argmin_results.append({
        'Book': book, 'Genre': genre, 'Form': form,
        'argmin_tau': argmin_tau, 'B_min': B_min,
        'B_start': B_start, 'B_end': B_end,
        'delta_B': B_end - B_start,
        'low_confidence': low_confidence,  # True if < 5 valid tau slices for this book
    })

df_argmin = pd.DataFrame(argmin_results)
df_argmin.to_csv(DIRS['stats'] / 'C_argmin_balance.csv', index=False)

# Statistical test: is argmin(B) non-uniform? KS test vs. Uniform
from scipy.stats import kstest, uniform
argmin_vals = df_argmin['argmin_tau'].dropna().values
ks_stat, ks_p = kstest(argmin_vals, uniform(0, 1).cdf)

print('=== argmin(B(τ)) Distribution ===')
print(f'  N books with valid argmin: {len(df_argmin)}')
print(f'  Median argmin: {np.median(argmin_vals):.3f}')
print(f'  Mode (histogram bin centre): {np.histogram(argmin_vals, bins=10)[1][np.histogram(argmin_vals, bins=10)[0].argmax()]:.2f}')
print(f'  KS test vs. Uniform(0,1): stat={ks_stat:.4f}, p={ks_p:.6f}')
print(f'  Interpretation: {"NON-UNIFORM" if ks_p < 0.05 else "cannot reject uniform"}')
print('\nMedian delta_B (B_end - B_start) by genre:')
print(df_argmin.groupby('Genre')['delta_B'].median().reindex(GENRE_ORDER).round(3).to_string())

In [ ]:
# BLOCK C5: Figure — argmin(B(t)) histograms by genre

genres_present = [g for g in GENRE_ORDER if g in df_argmin['Genre'].values]
ncols = 4
nrows = int(np.ceil(len(genres_present) / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(14, 3.5 * nrows), sharex=True, sharey=False)
axes_flat = axes.flatten()

for i, genre in enumerate(genres_present):
    ax = axes_flat[i]
    vals = df_argmin[df_argmin['Genre'] == genre]['argmin_tau'].dropna()
    color = GENRE_COLORS.get(genre, 'grey')
    ax.hist(vals, bins=10, range=(0, 1), color=color, edgecolor='white', linewidth=0.5)
    ax.axvline(vals.median(), color='black', linewidth=1.5, linestyle='--', label=f'Median={vals.median():.2f}')
    ax.axvspan(0.60, 0.75, alpha=0.1, color='red', label='Freytag zone (0.6–0.75)')
    ax.set_title(f'{GENRE_LABELS_SHORT.get(genre, genre)}\n(n={len(vals)})', fontsize=11)
    ax.set_xlim(0, 1)
    if i % ncols == 0:
        ax.set_ylabel('Count')
    ax.legend(fontsize=7, frameon=False)

for j in range(len(genres_present), len(axes_flat)):
    axes_flat[j].set_visible(False)

fig.suptitle('When Does Social Tension Peak? Narrative Position of Maximum Imbalance by Genre\n'
'(Freytag climax prediction: 60–75% through the narrative)', fontsize=18, fontweight='bold', y=1.01)
fig.text(0.5, -0.01, 'Normalised narrative time τ', ha='center', fontsize=14)
plt.tight_layout()
savefig(fig, 'figs_C', 'C3_argmin_balance_by_genre.png')
savefig(fig, 'figs_C', 'C3_argmin_balance_by_genre.pdf')

In [ ]:
# BLOCK C6: Figure — Triad type fractions over narrative time
# Shows exactly which triad types dominate in each genre

fig, axes = plt.subplots(2, 4, figsize=(16, 10), sharex=True, sharey=True)
axes_flat = axes.flatten()

triad_colors = {'f_ppp': '#2ca02c', 'f_ppn': '#ff7f0e', 'f_pnn': '#1f77b4', 'f_nnn': '#d62728'}
triad_labels = {'f_ppp': '+++  (all friends)', 'f_ppn': '++-  (imbalanced)',
                'f_pnn': '+--  (common enemy)', 'f_nnn': '---  (all enemies)'}

for i, genre in enumerate(genres_present):
    ax = axes_flat[i]
    gdf = df_balance[df_balance['Genre'] == genre]
    for metric in ['f_ppp', 'f_ppn', 'f_pnn', 'f_nnn']:
        res = genre_mean_ci(gdf, metric, genres=[genre])
        if genre in res:
            ax.plot(TAU_BINS, res[genre]['mean'], color=triad_colors[metric], linewidth=2, label=triad_labels[metric])
    color = GENRE_COLORS.get(genre, 'grey')
    ax.set_facecolor((*matplotlib.colors.to_rgb(color), 0.04))
    ax.set_title(GENRE_LABELS_SHORT.get(genre, genre), fontsize=16)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    if i % 4 == 0:
        ax.set_ylabel('Triad fraction')
    if i >= 4:
        ax.set_xlabel('τ')

handles = [mpatches.Patch(color=c, label=l) for c, l in
           zip(triad_colors.values(), triad_labels.values())]
fig.legend(handles=handles, loc='lower center', ncol=4, frameon=False,
           bbox_to_anchor=(0.5, -0.02), fontsize=16)
fig.suptitle('Triad Type Fractions Over Narrative Time by Genre\n'
             '(Balanced triads: +++ and +--; Imbalanced: ++- and ---)', fontsize=22, fontweight='bold', y=1.01)
plt.tight_layout()
savefig(fig, 'figs_C', 'C4_triad_fractions_by_genre.png')
savefig(fig, 'figs_C', 'C4_triad_fractions_by_genre.pdf')

In [ ]:
# BLOCK C7: Statistical tests for balance hypotheses

from scipy.stats import kruskal, mannwhitneyu

balance_stats = []

# H4: Genre predicts argmin(B) distribution
groups_argmin = [df_argmin[df_argmin['Genre'] == g]['argmin_tau'].dropna().values
                 for g in genres_present]
groups_argmin = [g for g in groups_argmin if len(g) >= 3]
H, p = kruskal(*groups_argmin)
print(f'H4 Kruskal-Wallis (argmin_tau ~ Genre): H={H:.3f}, p={p:.6f}')
balance_stats.append({'test': 'KW_argmin_by_genre', 'H': H, 'p': p})

# H5-H8: Pairwise genre tests on delta_B (B_end - B_start). Comedy (U-shape: delta_B ~ 0 or slightly positive). Tragedy (collapse: delta_B strongly negative)
delta_by_genre = {g: df_argmin[df_argmin['Genre'] == g]['delta_B'].dropna().values
                  for g in genres_present}

print('\nMedian delta_B = B_end - B_start (positive = balance increases toward end):')
for g, vals in delta_by_genre.items():
    if len(vals) > 0:
        print(f'  {g}: median={np.median(vals):.3f}, n={len(vals)}')

# Compare Tragedy vs Comedy in delta_B
if 'Tragedy / Drama' in delta_by_genre and 'Comedy / Satire' in delta_by_genre:
    u, p_tc = mannwhitneyu(delta_by_genre['Tragedy / Drama'], delta_by_genre['Comedy / Satire'], alternative='less')
    print(f'\nH5/H6 Mann-Whitney (Tragedy delta_B < Comedy delta_B): U={u:.0f}, p={p_tc:.6f}')
    balance_stats.append({'test': 'MWU_tragedy_vs_comedy_deltaB', 'U': u, 'p': p_tc})

pd.DataFrame(balance_stats).to_csv(DIRS['stats'] / 'C_balance_hypothesis_tests.csv', index=False)
print('\n Block C7 complete')

---
## Block D — Temporal Motifs

In [ ]:
# BLOCK D1: Character Introduction Rate Curve

char_intro_results = genre_mean_ci(df_temporal, 'f_chars_intro')
plot_genre_trajectories(
    char_intro_results,
    metric_label='Fraction of full cast introduced by this point',
    title='When Are Characters Introduced? Cumulative Cast Introduction by Genre',
    filename='D1_character_intro_by_genre.png',
    fig_dir='figs_D'
)

# Compute introduction half-life τ₅₀ per book
intro_halflife = []
for book, bdf_b in df_temporal.groupby('Book'):
    valid = bdf_b.dropna(subset=['f_chars_intro', 'tau']).sort_values('tau')
    if len(valid) < 2:
        continue
    # Find first tau where f_chars_intro >= 0.5
    past_half = valid[valid['f_chars_intro'] >= 0.5]
    tau50 = past_half['tau'].iloc[0] if len(past_half) > 0 else 1.0
    genre = valid.iloc[0]['Genre']
    form  = valid.iloc[0]['Form']
    intro_halflife.append({'Book': book, 'Genre': genre, 'Form': form, 'tau50': tau50})

df_halflife = pd.DataFrame(intro_halflife)

fig, ax = plt.subplots(figsize=(11, 5))
data_hl = [df_halflife[df_halflife['Genre'] == g]['tau50'].dropna().values
           for g in GENRE_ORDER if g in df_halflife['Genre'].values]
genres_hl = [g for g in GENRE_ORDER if g in df_halflife['Genre'].values]

bp = ax.boxplot(data_hl, labels=[GENRE_LABELS_SHORT.get(g, g) for g in genres_hl], patch_artist=True, notch=False, medianprops=dict(color='black', linewidth=1.5), flierprops=dict(marker='o', markersize=3, alpha=0.4))
for patch, g in zip(bp['boxes'], genres_hl):
    patch.set_facecolor(GENRE_COLORS[g])
    patch.set_alpha(0.85)
ax.axhline(0.5, color='grey', linestyle=':', linewidth=1)
ax.set_ylabel(r'Narrative position when half the cast has been introduced', fontsize=12)
ax.set_title('Cast Introduction Half-Life by Genre\n'
             'Lower = most characters appear early | Higher = characters introduced later', fontweight='bold')
ax.tick_params(axis='x', labelsize=8)
plt.tight_layout()
savefig(fig, 'figs_D', 'D1b_intro_halflife_by_genre.png')
savefig(fig, 'figs_D', 'D1b_intro_halflife_by_genre.pdf')

In [ ]:
# BLOCK D2: Edge Polarity Shift Dynamics

# For each character pair recurring in >=2 chapters, track polarity sequence
polarity_transitions = []
volatility_per_book = []

for book, bdf_b in df.groupby('Book'):
    genre = bdf_b['Genre_Primary'].iloc[0]
    form  = bdf_b['Form'].iloc[0]

    # Pairs recurring in >=2 chapters
    pair_groups = bdf_b.sort_values('Chapter').groupby('pair')
    volatile_count = 0
    total_recurring = 0

    for pair, pdf in pair_groups:
        if len(pdf) < 2:
            continue
        total_recurring += 1
        rels = pdf['Relationship'].tolist()
        if len(set(rels)) > 1:
            volatile_count += 1
        # Consecutive transitions
        for i in range(len(rels) - 1):
            polarity_transitions.append({
                'Book': book, 'Genre': genre, 'Form': form,
                'from': rels[i], 'to': rels[i+1]
            })

    volatility = volatile_count / total_recurring if total_recurring > 0 else np.nan
    volatility_per_book.append({
        'Book': book, 'Genre': genre, 'Form': form,
        'volatility': volatility,
        'n_recurring_pairs': total_recurring,
    })

df_transitions = pd.DataFrame(polarity_transitions)
df_volatility  = pd.DataFrame(volatility_per_book)



# BLOCK D2b: Edge Persistence & Burstiness

# Temporal network analysis extension. Edge persistence: how many consecutive chapters does each edge survive? Burstiness: are inter-appearance intervals clustered or regular?

import numpy as np

persistence_results = []

for book, bdf_b in df.groupby('Book'):
    genre  = bdf_b['Genre_Primary'].iloc[0]
    form   = bdf_b['Form'].iloc[0]

    # Sorted chapter sequence for THIS book (handles non-integer/non-contiguous chapters)
    chapters = sorted(bdf_b['Chapter'].unique())
    chapter_to_idx = {ch: i for i, ch in enumerate(chapters)}  # position index
    n_chapters = len(chapters)

    for pair, pdf in bdf_b.groupby('pair'):
        pair_chapters = sorted(pdf['Chapter'].unique())
        if len(pair_chapters) < 2:
            continue  # need at least 2 appearances for any temporal metric

        # Map to positional indices (robust to any chapter numbering scheme)
        pair_idxs = [chapter_to_idx[ch] for ch in pair_chapters]

        # Edge persistence: consecutive runs
        runs = []
        run_len = 1
        for i in range(1, len(pair_idxs)):
            if pair_idxs[i] == pair_idxs[i-1] + 1:
                run_len += 1
            else:
                runs.append(run_len)
                run_len = 1
        runs.append(run_len)

        # Burstiness of inter-appearance intervals. Inter-event times in chapter-position units
        inter_event = [pair_idxs[i] - pair_idxs[i-1] for i in range(1, len(pair_idxs))]
        iet_arr = np.array(inter_event, dtype=float)
        mu_iet = iet_arr.mean()
        sigma_iet = iet_arr.std()
        # Burstiness coefficient: -1 = perfectly regular, 0 = Poisson, +1 = maximally bursty
        burstiness = (sigma_iet - mu_iet) / (sigma_iet + mu_iet) if (sigma_iet + mu_iet) > 0 else np.nan

        # Final polarity (last observed, sorted by chapter)
        final_polarity = pdf.sort_values('Chapter').iloc[-1]['Relationship']

        persistence_results.append({
            'Book':             book,
            'Genre':            genre,
            'Form':             form,
            'pair':             pair,
            'final_polarity':   final_polarity,
            'mean_run_length':  np.mean(runs),
            'max_run_length':   max(runs),
            'n_runs':           len(runs),
            'n_appearances':    len(pair_chapters),
            'total_chapters':   n_chapters,
            'coverage':         len(pair_chapters) / n_chapters,
            'burstiness':       burstiness,
            'mean_iet':         mu_iet,
        })

df_persistence = pd.DataFrame(persistence_results)
df_persistence.to_csv(DIRS['tables'] / 'D2b_edge_persistence_burstiness.csv', index=False)

# Summary tables

print("=== Edge Persistence by Genre × Polarity (mean consecutive-chapter run length) ===")
print(
    df_persistence.groupby(['Genre', 'final_polarity'])['mean_run_length']
    .mean()
    .unstack(fill_value=np.nan)
    .reindex(GENRE_ORDER)
    .round(2)
    .to_string()
)

print("\n=== Burstiness by Genre × Polarity (κ: +1=bursty, -1=regular) ===")
print(
    df_persistence.groupby(['Genre', 'final_polarity'])['burstiness']
    .mean()
    .unstack(fill_value=np.nan)
    .reindex(GENRE_ORDER)
    .round(3)
    .to_string()
)

# Statistical tests
from scipy.stats import mannwhitneyu, kruskal

persistence_stats = []

# H: Positive edges persist longer than negative edges (across all books)
pos_runs = df_persistence[df_persistence['final_polarity'] == 'positive']['mean_run_length'].dropna()
neg_runs = df_persistence[df_persistence['final_polarity'] == 'negative']['mean_run_length'].dropna()
if len(pos_runs) > 0 and len(neg_runs) > 0:
    u_stat, p_val = mannwhitneyu(pos_runs, neg_runs, alternative='greater')
    print(f"\nH_persist: Positive edges have longer runs than negative — U={u_stat:.0f}, p={p_val:.4f}")
    persistence_stats.append({'test': 'MWU_pos_vs_neg_persistence', 'U': u_stat, 'p': p_val, 'median_pos': pos_runs.median(), 'median_neg': neg_runs.median()})

# H: Tragedy edges are shorter-lived than Comedy edges
trag_runs = df_persistence[df_persistence['Genre'] == 'Tragedy / Drama']['mean_run_length'].dropna()
com_runs  = df_persistence[df_persistence['Genre'] == 'Comedy / Satire']['mean_run_length'].dropna()
if len(trag_runs) > 0 and len(com_runs) > 0:
    u_stat2, p_val2 = mannwhitneyu(trag_runs, com_runs, alternative='less')
    print(f"H_trag: Tragedy edges shorter-lived than Comedy — U={u_stat2:.0f}, p={p_val2:.4f}")
    persistence_stats.append({'test': 'MWU_tragedy_vs_comedy_persistence', 'U': u_stat2, 'p': p_val2})

# H: Antagonistic edges are more bursty than positive edges
pos_burst = df_persistence[df_persistence['final_polarity'] == 'positive']['burstiness'].dropna()
neg_burst = df_persistence[df_persistence['final_polarity'] == 'negative']['burstiness'].dropna()
if len(neg_burst) > 0 and len(pos_burst) > 0:
    u_stat3, p_val3 = mannwhitneyu(neg_burst, pos_burst, alternative='greater')
    print(f"H_burst: Negative edges more bursty than positive — U={u_stat3:.0f}, p={p_val3:.4f}")
    persistence_stats.append({'test': 'MWU_neg_vs_pos_burstiness', 'U': u_stat3, 'p': p_val3})

pd.DataFrame(persistence_stats).to_csv(DIRS['stats'] / 'D2b_persistence_hypothesis_tests.csv', index=False)
print("\n Block D2b complete — df_persistence saved")

print(f'Total polarity transitions: {len(df_transitions):,}')
print(f'Overall edge volatility: {df_volatility["volatility"].mean():.1%}')
print(f'\nVolatility by genre:')
print(df_volatility.groupby('Genre')['volatility'].mean().reindex(GENRE_ORDER).round(3).to_string())

In [ ]:
# Transition matrices heatmap by genre

from_vals = ['positive', 'negative', 'neutral']
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes_flat = axes.flatten()

for i, genre in enumerate(genres_present):
    ax = axes_flat[i]
    gdf_t = df_transitions[df_transitions['Genre'] == genre]
    if len(gdf_t) == 0:
        ax.set_visible(False)
        continue

    trans_mat = gdf_t.groupby(['from', 'to']).size().unstack(fill_value=0)
    trans_mat = trans_mat.reindex(index=from_vals, columns=from_vals, fill_value=0)
    # Row-normalize
    row_sums = trans_mat.sum(axis=1).replace(0, 1)
    trans_norm = trans_mat.div(row_sums, axis=0)

    sns.heatmap(trans_norm, ax=ax, annot=True, fmt='.2f', cmap='RdYlGn', vmin=0, vmax=1, cbar=False, linewidths=0.5, xticklabels=['pos', 'neg', 'neu'], yticklabels=['pos', 'neg', 'neu'])
    n_trans = len(gdf_t)
    ax.set_title(f'{GENRE_LABELS_SHORT.get(genre, genre)}\n(n={n_trans} transitions)', fontsize=12)
    ax.set_xlabel('To', fontsize=12)
    ax.set_ylabel('From', fontsize=12)

for j in range(len(genres_present), len(axes_flat)):
    axes_flat[j].set_visible(False)

fig.suptitle('Edge Polarity Transition Matrices by Genre\n'
             'Row-normalised probabilities: P(to | from)', fontsize=18, fontweight='bold', y=1.01)
plt.tight_layout()
savefig(fig, 'figs_D', 'D2_polarity_transitions_by_genre.png')
savefig(fig, 'figs_D', 'D2_polarity_transitions_by_genre.pdf')

In [ ]:
# BLOCK D2c: Visualise persistence and burstiness

fig, axes = plt.subplots(1, 2, figsize=(14, 7))

# Panel 1: Mean run length by polarity × genre
pol_order = ['positive', 'neutral', 'negative']
df_pers_plot = (
    df_persistence[df_persistence['Genre'].isin(GENRE_ORDER)]
    .groupby(['Genre', 'final_polarity'])['mean_run_length']
    .mean()
    .reset_index()
)

for i, pol in enumerate(pol_order):
    sub = df_pers_plot[df_pers_plot['final_polarity'] == pol]
    sub = sub.set_index('Genre').reindex(GENRE_ORDER)
    axes[0].barh(
        [GENRE_LABELS_SHORT.get(g, g)+ f'  ({pol[0]})' for g in GENRE_ORDER],
        sub['mean_run_length'].fillna(0),
        color=POLARITY_COLORS[pol], alpha=0.75,
        label=pol.capitalize()
    )
axes[0].set_xlabel('Mean consecutive-chapter run length')
axes[0].set_title('Edge Persistence by Genre & Polarity\n(how many chapters does an edge survive?)', fontsize=18, fontweight='bold')
axes[0].legend(frameon=False, fontsize=10, loc='lower right')

# Panel 2: Burstiness distribution by polarity
for pol in pol_order:
    sub = df_persistence[df_persistence['final_polarity'] == pol]['burstiness'].dropna()
    axes[1].hist(sub, bins=30, alpha=0.6, label=pol.capitalize(),
                 color=POLARITY_COLORS[pol], density=True)
axes[1].axvline(0, color='black', linestyle='--', linewidth=1, label='Poisson baseline')
axes[1].set_xlabel('Burstiness coefficient B')
axes[1].set_ylabel('Density')
axes[1].set_title('Inter-appearance Burstiness by Polarity\n(B>0: bursty, B<0: regular)', fontsize=18, fontweight='bold')
axes[1].legend(frameon=False, fontsize=10, loc='upper right')

plt.tight_layout()
savefig(fig, 'figs_D', 'D2b_persistence_burstiness.png')
savefig(fig, 'figs_D', 'D2b_persistence_burstiness.pdf')

--- 
## Block D_MOTIF — Temporal Interaction Motif Analysis
3-event signed sequences on recurring character pairs. This is temporal motif analysis (event-sequence level), complementing the static SBT triad-type analysis in Block C.


In [ ]:
# BLOCK D_MOTIF: Temporal Interaction Motif Analysis

# A temporal motif here is a 3-event polarity sequence on a single
# character pair across consecutive chapter appearances.
# Examples:
#   +++  stable alliance
#   ---  stable enmity
#   +--  alliance collapse (betrayal)
#   --+  reconciliation after conflict
#   +-+  betrayal followed by reconciliation
#   -+-  truce-relapse

# Methodology: for every character pair with >= 3 chapter appearances,
# extract all overlapping windows of 3 consecutive polarity observations.
# Neutral observations are excluded (only +/- sequences counted).
# Counts are normalised within genre by total valid 3-event sequences.

pol_sym = {'positive': '+', 'negative': '-'}  # neutral excluded

MOTIF_READABLE = {
    '+++': 'Stable alliance (+++)',
    '---': 'Stable enmity (---)',
    '+--': 'Alliance collapse (+--)',
    '--+': 'Reconciliation (--+)',
    '+-+': 'Betrayal-reconciliation (+-+)',
    '-+-': 'Truce-relapse (-+-)',
    '++-': 'Drift to conflict (++-)',
    '-++': 'Drift to alliance (-++)',
}

motif_rows = []
for book, bdf_b in df.groupby('Book'):
    genre = bdf_b['Genre_Primary'].iloc[0]
    form  = bdf_b['Form'].iloc[0]

    pair_groups = bdf_b.sort_values('Chapter').groupby('pair')
    for pair, pdf in pair_groups:
        rels_all = pdf['Relationship'].tolist()
        # Keep only non-neutral and convert to symbol
        rels_sig = [pol_sym[r] for r in rels_all if r in pol_sym]
        if len(rels_sig) < 3:
            continue
        # Sliding window of 3
        for i in range(len(rels_sig) - 2):
            triplet = ''.join(rels_sig[i:i+3])
            motif_rows.append({'Book': book, 'Genre': genre, 'Form': form, 'motif': triplet})

df_motifs = pd.DataFrame(motif_rows)
print(f'Total 3-event motif observations: {len(df_motifs):,}')
print(f'Books contributing: {df_motifs["Book"].nunique()}')

# Compute fractions per genre
motif_counts = (df_motifs.groupby(['Genre', 'motif']).size()
                .reset_index(name='count'))
motif_totals = motif_counts.groupby('Genre')['count'].transform('sum')
motif_counts['fraction'] = motif_counts['count'] / motif_totals

motif_pivot = (motif_counts.pivot(index='Genre', columns='motif', values='fraction')
               .fillna(0).reindex(GENRE_ORDER).dropna(how='all'))

# Identify most discriminative motifs by cross-genre standard deviation
top_motifs = motif_pivot.std().sort_values(ascending=False).head(6).index.tolist()
print(f'\nMost discriminative motifs (highest cross-genre std):')
print(motif_pivot[top_motifs].round(3).to_string())

motif_pivot.to_csv(DIRS['stats'] / 'D_temporal_motifs.csv')

# Figure: grouped bar chart of top-6 discriminative motifs by genre
fig, ax = plt.subplots(figsize=(13, 5))
x      = np.arange(len(top_motifs))
n_gen  = len(GENRE_ORDER)
w      = 0.08
offsets = np.linspace(-(n_gen-1)*w/2, (n_gen-1)*w/2, n_gen)

for i, genre in enumerate(GENRE_ORDER):
    if genre not in motif_pivot.index:
        continue
    vals  = motif_pivot.loc[genre, top_motifs].values
    short = GENRE_LABELS_SHORT.get(genre, genre)
    ax.bar(x + offsets[i], vals, width=w * 0.92, color=GENRE_COLORS[genre], label=short, alpha=0.88)

ax.set_xticks(x)
ax.set_xticklabels([MOTIF_READABLE.get(m, m) for m in top_motifs], fontsize=10, rotation=15, ha='right')
ax.set_ylabel('Fraction of 3-event sequences')
ax.set_title('Dominant Temporal Interaction Motifs by Genre', fontsize=18, fontweight='bold')
ax.legend(frameon=False, fontsize=8, ncol=4, loc='upper right')
plt.tight_layout()
savefig(fig, 'figs_D', 'D_motif_analysis.png')
savefig(fig, 'figs_D', 'D_motif_analysis.pdf')

In [ ]:
# BLOCK D3: Network Densification Exponent 

# log(E) ~ alpha * log(N) + c  over narrative time

from scipy.stats import linregress

densification = []
for book, bdf_b in df_temporal.groupby('Book'):
    valid = bdf_b[(bdf_b['N_t'] > 1) & (bdf_b['E_t'] > 0)].copy()
    if len(valid) < 4:
        continue
    log_N = np.log(valid['N_t'].values)
    log_E = np.log(valid['E_t'].values)
    if len(np.unique(log_N)) < 2:  
        continue
    slope, intercept, r, p, se = linregress(log_N, log_E)
    densification.append({
        'Book': book, 'Genre': valid.iloc[0]['Genre'],
        'Form': valid.iloc[0]['Form'],
        'alpha': slope, 'r_squared': r**2, 'p_slope': p,
    })

df_densif = pd.DataFrame(densification)
print(f'Densification exponent (alpha) summary:')
print(df_densif['alpha'].describe().round(3))
print(f'\nFraction with alpha > 1 (superlinear): {(df_densif["alpha"] > 1).mean():.1%}')
print(f'\nAlpha by genre (median):')
print(df_densif.groupby('Genre')['alpha'].median().reindex(GENRE_ORDER).round(3).to_string())

fig, ax = plt.subplots(figsize=(11, 5))
data_alpha = [df_densif[df_densif['Genre'] == g]['alpha'].dropna().values
              for g in GENRE_ORDER if g in df_densif['Genre'].values]
genres_alpha = [g for g in GENRE_ORDER if g in df_densif['Genre'].values]
bp = ax.boxplot(data_alpha, labels=[GENRE_LABELS_SHORT.get(g, g) for g in genres_alpha], patch_artist=True, medianprops=dict(color='black', linewidth=1.5),
                flierprops=dict(marker='o', markersize=3, alpha=0.4))
for patch, g in zip(bp['boxes'], genres_alpha):
    patch.set_facecolor(GENRE_COLORS[g]); patch.set_alpha(0.85)
ax.axhline(1.0, color='#d62728', linestyle='--', linewidth=1.5, label='α=1 (linear)')
ax.set_ylabel('Densification exponent α')
ax.set_title('Network Densification Exponent (α) by Genre\n'
             'α > 1: superlinear growth (more relationships added per new character)', fontweight='bold')
ax.legend(frameon=False)
ax.tick_params(axis='x', labelsize=11)
plt.tight_layout()
savefig(fig, 'figs_D', 'D3_densification_exponent.png')
savefig(fig, 'figs_D', 'D3_densification_exponent.pdf')

In [ ]:
# BLOCK D4: Protagonist Centrality Trajectory 

# Protagonist = character with highest cumulative degree

# Track protagonist's (highest-degree character) centrality over narrative time
def compute_protagonist_trajectory(book, bdf):
    genre = bdf['Genre_Primary'].iloc[0]
    form  = bdf['Form'].iloc[0]

    chapters, edge_dicts, graphs = build_cumulative_networks(bdf, return_graphs=True)
    K = max(chapters)

    # Identify protagonist from FINAL network (highest cumulative degree)
    G_final = graphs[-1]
    if G_final.number_of_nodes() < 2:
        return pd.DataFrame()
    protagonist = max(dict(G_final.degree()).items(), key=lambda x: x[1])[0]

    rows = []
    for ch, G in zip(chapters, graphs):
        tau = ch / K
        if protagonist not in G.nodes():
            rows.append({'Book': book, 'Genre': genre, 'Form': form, 'chapter': ch, 'tau': tau, 'deg_centrality': 0, 'btw_centrality': 0})
            continue
        N = G.number_of_nodes()
        deg_c = G.degree(protagonist) / (N - 1) if N > 1 else 0
        # Betweenness is expensive for large networks; limit to < 30 nodes
        if N <= 30:
            btw = nx.betweenness_centrality(G).get(protagonist, 0)
        else:
            btw = np.nan
        rows.append({'Book': book, 'Genre': genre, 'Form': form, 'chapter': ch, 'tau': tau, 'deg_centrality': deg_c, 'btw_centrality': btw})
    return pd.DataFrame(rows)

log.info('Computing protagonist trajectories...')
t0 = time.time()
prot_results = []
for i, book in enumerate(books_list):
    bdf = df[df['Book'] == book]
    result = compute_protagonist_trajectory(book, bdf)
    if len(result) > 0:
        prot_results.append(result)

df_prot = pd.concat(prot_results, ignore_index=True)
log.info(f'Protagonist trajectories done in {time.time()-t0:.0f}s')

# Plot degree centrality trajectory by genre
prot_genre_res = genre_mean_ci(df_prot, 'deg_centrality')
plot_genre_trajectories(
    prot_genre_res,
    metric_label='Protagonist degree centrality',
    title='Protagonist Degree Centrality Over Narrative Time by Genre\n'
          '(Protagonist = character with highest cumulative degree)',
    filename='D4_protagonist_centrality_by_genre.png',
    fig_dir='figs_D'
)

In [ ]:
# BLOCK D5: Negative Edge Concentration (Gini)

# Gini of negative-edge degree distribution

# Gini coefficient of array x (>= 0).
def gini(x):
    x = np.sort(np.abs(x))
    n = len(x)
    if n == 0 or x.sum() == 0:
        return np.nan
    idx = np.arange(1, n + 1)
    return (2 * np.dot(idx, x) / (n * x.sum())) - (n + 1) / n

def compute_neg_gini_trajectory(book, bdf):
    genre = bdf['Genre_Primary'].iloc[0]
    form  = bdf['Form'].iloc[0]
    chapters, edge_dicts, graphs = build_cumulative_networks(bdf, return_graphs=True)
    K = max(chapters)
    rows = []
    for ch, G in zip(chapters, graphs):
        tau = ch / K
        neg_degree = {n: 0 for n in G.nodes()}
        for a, b, d in G.edges(data=True):
            if d.get('polarity') == 'negative':
                neg_degree[a] = neg_degree.get(a, 0) + 1
                neg_degree[b] = neg_degree.get(b, 0) + 1
        neg_vals = np.array(list(neg_degree.values()))
        g = gini(neg_vals) if neg_vals.sum() > 0 else np.nan
        rows.append({'Book': book, 'Genre': genre, 'Form': form,
                     'chapter': ch, 'tau': tau, 'neg_gini': g})
    return pd.DataFrame(rows)

log.info('Computing negative edge Gini...')
t0 = time.time()
gini_results = [compute_neg_gini_trajectory(book, df[df['Book'] == book])
                for book in books_list]
df_gini = pd.concat(gini_results, ignore_index=True)
log.info(f'Gini done in {time.time()-t0:.0f}s')

gini_genre_res = genre_mean_ci(df_gini, 'neg_gini')
plot_genre_trajectories(
    gini_genre_res,
    metric_label='Gini coefficient of negative-edge degree',
    title='Conflict Localisation Over Narrative Time by Genre\n'
          'High Gini = conflict concentrated on few characters (villain dynamic)\n'
          'Low Gini = distributed conflict (war/political narrative)',
    filename='D5_conflict_gini_by_genre.png',
    fig_dir='figs_D'
)

In [ ]:
# BLOCK D6: Social Entropy Trajectory 
# Already computed in Block B1 as H_deg — just plot here

entropy_results = genre_mean_ci(df_temporal, 'H_deg')
plot_genre_trajectories(
    entropy_results,
    metric_label='Shannon entropy of degree distribution H_deg(τ)',
    title='Social Entropy Over Narrative Time by Genre\n'
          'High H = democratic network | Low H = star/protagonist-dominated',
    filename='D6_social_entropy_by_genre.png',
    fig_dir='figs_D'
)

In [ ]:
# BLOCK D7: Community Modularity Q(t) 

# Run Louvain 10x per timestep, average Q(t). Limited to N_t >= 5; computationally expensive

from networkx.algorithms.community import greedy_modularity_communities

def compute_modularity_trajectory(book, bdf):
    genre = bdf['Genre_Primary'].iloc[0]
    form  = bdf['Form'].iloc[0]
    chapters, edge_dicts, graphs = build_cumulative_networks(bdf, return_graphs=True)
    K = max(chapters)
    rows = []
    for ch, G in zip(chapters, graphs):
        tau = ch / K
        N = G.number_of_nodes()
        if N < 5 or G.number_of_edges() == 0:
            rows.append({'Book': book, 'Genre': genre, 'Form': form, 'chapter': ch, 'tau': tau, 'modularity': np.nan})
            continue
        try:
            communities = greedy_modularity_communities(G)
            Q = nx.community.modularity(G, communities)
        except Exception:
            Q = np.nan
        rows.append({'Book': book, 'Genre': genre, 'Form': form, 'chapter': ch, 'tau': tau, 'modularity': Q})
    return pd.DataFrame(rows)

log.info('Computing modularity trajectories (may be slow)...')
t0 = time.time()
mod_results = [compute_modularity_trajectory(book, df[df['Book'] == book])
               for book in books_list]
df_mod = pd.concat(mod_results, ignore_index=True)
log.info(f'Modularity done in {time.time()-t0:.0f}s')

mod_genre_res = genre_mean_ci(df_mod, 'modularity')
plot_genre_trajectories(
    mod_genre_res,
    metric_label='Network modularity Q(τ)',
    title='Community Modularity Over Narrative Time by Genre\n'
          'High Q = distinct social groups | Low Q = integrated network',
    filename='D7_modularity_by_genre.png',
    fig_dir='figs_D'
)

---
## Block E — Genre Classification
Classification + feature importance

In [ ]:
# BLOCK E1: Feature Engineering — Build per-book feature vectors

K_BINS = 10  # 10 temporal bins → bin centres at {0.05, 0.15, ..., 0.95}
bin_edges = np.linspace(0, 1, K_BINS + 1)
bin_centres = (bin_edges[:-1] + bin_edges[1:]) / 2

TEMPORAL_METRICS = ['density', 'd_pos', 'd_neg', 'r_plus', 'H_deg', 'C_avg']
BALANCE_METRICS  = ['B']

# Merge temporal metrics with balance
df_all_temp = df_temporal.merge(
    df_balance[['Book', 'chapter', 'tau', 'B', 'B_weak', 'f_ppn', 'f_ppp', 'f_nnn']],
    on=['Book', 'chapter', 'tau'], how='left'
)
all_metrics = TEMPORAL_METRICS + BALANCE_METRICS

# Build feature vector for one book from its temporal metrics
def build_feature_vector(book, book_df):
    feats = {}

    for metric in all_metrics:
        series = book_df.dropna(subset=[metric, 'tau']).sort_values('tau')
        vals = series[metric].values
        taus = series['tau'].values
        if len(vals) < 2:
            # All features for this metric = NaN
            for k in range(K_BINS):
                feats[f'{metric}_bin{k}'] = np.nan
            for stat in ['mean','slope','var','argmin','argmax','start','end','delta','range']:
                feats[f'{metric}_{stat}'] = np.nan
            continue

        # Approach A: binned means
        for k, (lo, hi) in enumerate(zip(bin_edges[:-1], bin_edges[1:])):
            mask = (taus >= lo) & (taus < hi) if k < K_BINS - 1 else (taus >= lo)
            feats[f'{metric}_bin{k}'] = np.nanmean(vals[mask]) if mask.any() else np.nan

        # Approach B: summary statistics
        from scipy.stats import linregress
        if len(vals) >= 2:
            slope = linregress(taus, vals).slope
        else:
            slope = np.nan
        feats[f'{metric}_mean']   = np.nanmean(vals)
        feats[f'{metric}_slope']  = slope
        feats[f'{metric}_var']    = np.nanvar(vals)
        feats[f'{metric}_argmin'] = taus[np.nanargmin(vals)] if len(vals) > 0 else np.nan
        feats[f'{metric}_argmax'] = taus[np.nanargmax(vals)] if len(vals) > 0 else np.nan
        feats[f'{metric}_start']  = vals[0]
        feats[f'{metric}_end']    = vals[-1]
        feats[f'{metric}_delta']  = vals[-1] - vals[0]
        feats[f'{metric}_range']  = np.nanmax(vals) - np.nanmin(vals)

    return feats

log.info('Building feature matrix...')
feature_rows = []
for book, bdf_b in df_all_temp.groupby('Book'):
    fv = build_feature_vector(book, bdf_b)
    fv['Book']  = book
    fv['Genre'] = bdf_b['Genre'].iloc[0]
    fv['Form']  = bdf_b['Form'].iloc[0]
    fv['Author']= bdf_b['Author'].iloc[0]
    feature_rows.append(fv)

# Add scalar features from other analyses
df_scalar = df_halflife[['Book', 'tau50']].merge(
    df_volatility[['Book', 'volatility']], on='Book', how='outer'
).merge(
    df_densif[['Book', 'alpha']], on='Book', how='outer'
).merge(
    df_argmin[['Book', 'argmin_tau', 'delta_B']], on='Book', how='outer'
)

df_features = pd.DataFrame(feature_rows)
df_features = df_features.merge(df_scalar, on='Book', how='left')
df_features.to_csv(DIRS['tables'] / 'E_feature_matrix.csv', index=False)

print(f' Feature matrix: {df_features.shape[0]} books × {df_features.shape[1]} columns')
print(f'  Genre distribution in feature matrix:')
print(df_features['Genre'].value_counts().to_string())

In [ ]:
# BLOCK E2: Genre Classification — Random Forest (primary)

# Confusion matrix + feature importance

# Prepare X and y
meta_cols = ['Book', 'Genre', 'Form', 'Author']
feature_cols = [c for c in df_features.columns if c not in meta_cols]

valid_books = df_features.dropna(subset=['Genre'])
X_raw = valid_books[feature_cols].values.astype(float)
y     = np.array(valid_books['Genre'].tolist())

# Impute NaN values with column median
imputer = SimpleImputer(strategy='median')
X = imputer.fit_transform(X_raw)

# ── Baselines ──
print('=== CLASSIFICATION RESULTS ===')

cv = StratifiedKFold(n_splits=N_CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

for name, clf in [
    ('Majority class', DummyClassifier(strategy='most_frequent', random_state=RANDOM_STATE)),
    ('Stratified random', DummyClassifier(strategy='stratified', random_state=RANDOM_STATE)),
]:
    y_pred = cross_val_predict(clf, X, y, cv=cv)
    f1 = f1_score(y, y_pred, average='macro', zero_division=0)
    print(f'{name}: macro-F1 = {f1:.3f}')

# ── Logistic Regression (L1) ──
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
lr = LogisticRegression(C=0.1, penalty='l1', solver='saga', max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE)
y_pred_lr = cross_val_predict(lr, X_scaled, y, cv=cv)
f1_lr = f1_score(y, y_pred_lr, average='macro', zero_division=0)
print(f'Logistic Regression (L1): macro-F1 = {f1_lr:.3f}')

# ── Random Forest (primary model) ──
rf = RandomForestClassifier(n_estimators=500, class_weight='balanced',
                             random_state=RANDOM_STATE, n_jobs=-1, max_features='sqrt')
y_pred_rf = cross_val_predict(rf, X, y, cv=cv)
f1_rf = f1_score(y, y_pred_rf, average='macro', zero_division=0)
print(f'Random Forest (500 trees): macro-F1 = {f1_rf:.3f}')
print()

# Full classification report
report = classification_report(y, y_pred_rf, zero_division=0)
print('Random Forest — Full Classification Report:')
print(report)

# Save report
with open(DIRS['stats'] / 'E_rf_classification_report.txt', 'w') as f:
    f.write(report)

# ── Confusion Matrix Figure ──
fig, ax = plt.subplots(figsize=(10, 8))
labels = [g for g in GENRE_ORDER if g in set(y)]
short_labels = [GENRE_LABELS_SHORT.get(g, g) for g in labels]
cm = confusion_matrix(y, y_pred_rf, labels=labels, normalize='true')
sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues', ax=ax,
            xticklabels=short_labels, yticklabels=short_labels,
            vmin=0, vmax=1, linewidths=0.5, cbar_kws={'label': 'Fraction'})
ax.set_xlabel('Predicted genre', fontsize=11)
ax.set_ylabel('True genre', fontsize=11)
ax.set_title(f'Genre Classification Confusion Matrix\n'
             f'Random Forest (500 trees, balanced), 10-fold CV | macro-F1 = {f1_rf:.2f}',
             fontsize=18, fontweight='bold')
ax.tick_params(axis='x', rotation=45, labelsize=8)
ax.tick_params(axis='y', rotation=0, labelsize=8)
plt.tight_layout()
savefig(fig, 'figs_E', 'E1_confusion_matrix.png')
savefig(fig, 'figs_E', 'E1_confusion_matrix.pdf')

In [ ]:
# BLOCK E3: Feature Importance from Random Forest

# Fit RF on full data for feature importance
rf_full = RandomForestClassifier(n_estimators=500, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)
rf_full.fit(X, y)

importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf_full.feature_importances_
}).sort_values('importance', ascending=False)

importance_df = importance_df[importance_df['feature'].isin([c for c in df_features.columns if c not in meta_cols])]

top_n = 10
top_feats = importance_df.head(top_n)

# Colour by metric type
def feat_color(fname):
    if fname.startswith('B')         or 'balance' in fname.lower(): return '#6f1926'
    if fname.startswith('d_pos')     or fname.startswith('r_plus'): return '#2ca02c'
    if fname.startswith('d_neg'):    return '#d62728'
    if fname.startswith('density'):  return '#369acc'
    if fname.startswith('H_deg'):    return '#9656a2'
    if fname.startswith('C_avg'):    return '#f4895f'
    return '#888888'

colors = [feat_color(f) for f in top_feats['feature']]

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(range(len(top_feats)), top_feats['importance'], color=colors, edgecolor='white', linewidth=0.4)
ax.set_yticks(range(len(top_feats)))
ax.set_yticklabels(top_feats['feature'], fontsize=14)
ax.invert_yaxis()
ax.set_xlabel('Feature Importance (Mean Decrease in Impurity)')
ax.set_title(f'Top {top_n} Most Predictive Features for Genre Classification\n'
             f'Random Forest on {len(feature_cols)} temporal network features', fontweight='bold')

# Legend: which metric families are present
legend_map = {
    'Balance (B)': '#6f1926', 'Positive density': '#2ca02c',
    'Negative density': '#d62728', 'Overall density': '#369acc',
    'Entropy': '#9656a2', 'Clustering': '#f4895f', 'Other': '#888888'
}
handles = [mpatches.Patch(color=c, label=l) for l, c in legend_map.items()]
ax.legend(handles=handles, frameon=False, fontsize=12, loc='lower right')

plt.tight_layout()
savefig(fig, 'figs_E', 'E2_feature_importance.png')
savefig(fig, 'figs_E', 'E2_feature_importance.pdf')

importance_df.to_csv(DIRS['stats'] / 'E_feature_importances.csv', index=False)

In [ ]:
# BLOCK E4: t-SNE Embedding of Feature Space

tsne = TSNE(n_components=2, perplexity=30, max_iter=1500, random_state=RANDOM_STATE, metric='euclidean')
X_2d = tsne.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(11, 8))

for genre in GENRE_ORDER:
    mask = valid_books['Genre'].values == genre
    if mask.sum() == 0:
        continue
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1], color=GENRE_COLORS[genre], alpha=0.65, s=30, label=f'{GENRE_LABELS_SHORT.get(genre, genre)} (n={mask.sum()})', edgecolors='white', linewidth=0.3)

# Add genre centroids
for genre in GENRE_ORDER:
    mask = valid_books['Genre'].values == genre
    if mask.sum() == 0:
        continue
    cx, cy = X_2d[mask, 0].mean(), X_2d[mask, 1].mean()
    short = genre.split(' / ')[0][:12]
    ax.text(cx, cy, short, fontsize=8, fontweight='bold', ha='center', va='center', bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7, edgecolor='none'))

ax.set_xlabel('t-SNE dimension 1')
ax.set_ylabel('t-SNE dimension 2')
ax.set_title('t-SNE Embedding of Temporal Network Feature Space\n'
             '(Each point = one book; colored by primary genre)', fontsize=18, fontweight='bold')
ax.legend(frameon=False, fontsize=8, bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
savefig(fig, 'figs_E', 'E3_tsne_genre.png')
savefig(fig, 'figs_E', 'E3_tsne_genre.pdf')

In [ ]:
# BLOCK E5: Statistical significance — Kruskal-Wallis on all temporal metrics

temporal_kw_results = []
for metric in all_metrics + ['H_deg', 'C_avg']:
    if metric not in df_all_temp.columns:
        continue
    # Use mean per book as the unit of analysis
    per_book_mean = df_all_temp.groupby(['Book', 'Genre'])[metric].mean().reset_index()
    groups = [per_book_mean[per_book_mean['Genre'] == g][metric].dropna().values
              for g in GENRE_ORDER if g in per_book_mean['Genre'].values]
    groups = [g for g in groups if len(g) >= 3]
    if len(groups) < 2:
        continue
    H, p = kruskal(*groups)
    N_total = sum(len(g) for g in groups)
    k = len(groups)
    eta2 = max(0, (H - k + 1) / (N_total - k))
    temporal_kw_results.append({
        'metric': metric, 'H': round(H, 3), 'p_value': round(p, 8),
        'significant_p05': p < 0.05, 'significant_bonferroni': p < 0.05 / len(all_metrics),
        'eta_squared': round(eta2, 4)
    })

kw_temporal_df = pd.DataFrame(temporal_kw_results).sort_values('p_value')
kw_temporal_df.to_csv(DIRS['stats'] / 'E_kruskal_wallis_temporal.csv', index=False)

print('=== TABLE 2 — Kruskal-Wallis: Genre Differences in Temporal Metrics ===')
display(kw_temporal_df)

---
## Block E_PAIRWISE — Pairwise Genre Discriminability
Binary classifiers for every genre pair reveal *which* genres are
distinguishable from each other, complementing the 8-class macro-F1.


In [ ]:
# BLOCK E_PAIRWISE: Pairwise genre discriminability matrix

# For each pair of genres, train a binary Random Forest (5-fold CV) and record macro-F1. Near 0.5 = coin-flip; near 1.0 = distinguishable.
# Run AFTER Block E2 (uses X, valid_books, imputer, feature_cols).

genres_list = [g for g in GENRE_ORDER if g in valid_books['Genre'].values]
n_g = len(genres_list)
pw_f1 = np.full((n_g, n_g), np.nan)
np.fill_diagonal(pw_f1, 1.0)

# Convert Genre column to plain numpy ONCE — avoids ArrowExtensionArray indexing error
_all_genres_np = np.array(valid_books['Genre'].tolist())  # plain object array

for i, g1 in enumerate(genres_list):
    for j, g2 in enumerate(genres_list):
        if i >= j:
            continue

        mask = np.isin(_all_genres_np, [g1, g2])
        X_p  = X[mask]
        y_p  = _all_genres_np[mask]  # plain numpy array — sklearn-safe

        if len(np.unique(y_p)) < 2 or len(y_p) < 6:
            continue

        min_class = int(np.min(np.bincount(np.where(y_p == g1, 0, 1))))
        if min_class < 2:
            continue
        n_splits = max(2, min(5, min_class))

        cv_p = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
        rf_p = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)
        y_hat = cross_val_predict(rf_p, X_p, y_p, cv=cv_p)
        pw_f1[i, j] = pw_f1[j, i] = f1_score(y_hat, y_p, average='macro', zero_division=0)

df_pairwise = pd.DataFrame(pw_f1, index=genres_list, columns=genres_list)
df_pairwise.to_csv(DIRS['stats'] / 'E_pairwise_f1.csv')

print('=== Pairwise genre discriminability (binary macro-F1) ===')
print(df_pairwise.round(3).to_string())
print('\nMost distinguishable pairs (F1 > 0.55):')
for ii, g1 in enumerate(genres_list):
    for jj, g2 in enumerate(genres_list):
        if ii < jj and not np.isnan(pw_f1[ii, jj]) and pw_f1[ii, jj] > 0.55:
            print(f'  {GENRE_LABELS_SHORT.get(g1,g1)} vs {GENRE_LABELS_SHORT.get(g2,g2)}: F1={pw_f1[ii,jj]:.3f}')

short_labs = [GENRE_LABELS_SHORT.get(g, g) for g in genres_list]
fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(pw_f1, cmap='RdYlGn', vmin=0.1, vmax=0.9, aspect='auto')
plt.colorbar(im, ax=ax, label='Binary RF macro-F1')
for ii in range(n_g):
    for jj in range(n_g):
        v = pw_f1[ii, jj]
        if not np.isnan(v):
            tc = 'white' if (v < 0.25 or v > 0.72) else 'black'
            ax.text(jj, ii, f'{v:.2f}', ha='center', va='center', fontsize=12, color=tc)
ax.set_xticks(range(n_g)); ax.set_yticks(range(n_g))
ax.set_xticklabels(short_labs, rotation=45, ha='right', fontsize=14)
ax.set_yticklabels(short_labs, fontsize=14)
ax.set_title('Pairwise Genre Discriminability\n(Binary Random Forest macro-F1, 5-fold CV)', fontweight='bold', fontsize=18)
plt.tight_layout()
savefig(fig, 'figs_E', 'E_pairwise_genre_f1.png')
savefig(fig, 'figs_E', 'E_pairwise_genre_f1.pdf')

---
## Block E_NOVELS — Novels-Only Genre Classification
Removes the form confound (Tragedy/Drama and Comedy/Satire are predominantly plays).
Restricts to the 6 novel-dominated genres for a form-balanced comparison.


In [ ]:
# BLOCK E_NOVELS: Novels-only genre classification

# Removes form confound: Form=='Novel' + 6 novel-dominated genres.
# Run AFTER Block E2 (uses imputer, feature_cols, f1_rf).

NOVEL_GENRES = [
    'Literary / Realist Fiction', 'Coming-of-Age / Bildungsroman',
    'Fantasy / Adventure',        'Science Fiction / Dystopia',
    'Mystery / Thriller',         'Romance / Domestic Fiction',
]

# Build novels-only subset — reset index to avoid alignment issues
valid_novels = valid_books[
    (valid_books['Form'] == 'Novel') & (valid_books['Genre'].isin(NOVEL_GENRES))
].copy().reset_index(drop=True)

X_nov = imputer.transform(valid_novels[feature_cols].values.astype(float))

# CRITICAL: convert Genre to plain numpy to avoid ArrowExtensionArray error
y_nov = np.array(valid_novels['Genre'].tolist())

print(f'Novels-only sample: {len(valid_novels)} books, {len(np.unique(y_nov))} genres')
print(pd.Series(y_nov).value_counts().to_string())

min_class_nov = int(pd.Series(y_nov).value_counts().min())
n_splits_nov  = max(2, min(N_CV_FOLDS, min_class_nov))
print(f'\nUsing {n_splits_nov}-fold CV (min class size = {min_class_nov})')

cv_nov = StratifiedKFold(n_splits=n_splits_nov, shuffle=True, random_state=RANDOM_STATE)
rf_nov = RandomForestClassifier(n_estimators=500, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)
y_pred_nov = cross_val_predict(rf_nov, X_nov, y_nov, cv=cv_nov)
f1_nov = f1_score(y_nov, y_pred_nov, average='macro', zero_division=0)
print(f'\nNovels-only macro-F1 = {f1_nov:.3f}  (full-corpus {f1_rf:.3f})')
print(classification_report(y_nov, y_pred_nov, zero_division=0))

fig, ax = plt.subplots(figsize=(8, 7))
nov_labels = [g for g in GENRE_ORDER if g in NOVEL_GENRES and g in set(y_nov)]
cm_nov     = confusion_matrix(y_nov, y_pred_nov, labels=nov_labels, normalize='true')
short_nov  = [GENRE_LABELS_SHORT.get(l, l) for l in nov_labels]
sns.heatmap(cm_nov, annot=True, fmt='.2f', cmap='Blues', ax=ax, xticklabels=short_nov, yticklabels=short_nov, vmin=0, vmax=1, linewidths=0.5, cbar_kws={'label': 'Fraction'})
ax.set_xlabel('Predicted genre'); ax.set_ylabel('True genre')
ax.set_title(f'Genre Classification – Novels Only (6 genres)\nmacro-F1 = {f1_nov:.2f}', fontsize=15, fontweight='bold')
ax.tick_params(axis='x', rotation=45, labelsize=9)
plt.tight_layout()
savefig(fig, 'figs_E', 'E_novels_only_confusion.png')
savefig(fig, 'figs_E', 'E_novels_only_confusion.pdf')

---
## Block F — Epoch & Author Comparison

In [ ]:
# BLOCK F1: Balance B(t) by literary epoch

# Merge epoch onto balance data
df_balance_epoch = df_balance.merge(
    registry[['Book', 'Epoch']], on='Book', how='left'
)

fig, ax = plt.subplots(figsize=(12, 6))
EPOCH_COLORS_LIST = ['#7B3F00','#E65100','#F9A825','#2E7D32','#1565C0','#6A1B9A']

for epoch, color in zip(EPOCH_ORDER, EPOCH_COLORS_LIST):
    epoch_books = df_balance_epoch[df_balance_epoch['Epoch'] == epoch]['Book'].unique()
    if len(epoch_books) < 3:
        continue
    epoch_df = df_balance_epoch[df_balance_epoch['Epoch'] == epoch]
    traces = np.array([
        np.interp(TAU_BINS,
                  epoch_df[epoch_df['Book'] == b].dropna(subset=['B','tau']).sort_values('tau')['tau'].values,
                  epoch_df[epoch_df['Book'] == b].dropna(subset=['B','tau']).sort_values('tau')['B'].values)
        if len(epoch_df[epoch_df['Book'] == b].dropna(subset=['B'])) >= 2 else np.full(N_BINS, np.nan)
        for b in epoch_books
    ])
    mean_traj = np.nanmean(traces, axis=0)
    ax.plot(TAU_BINS, mean_traj, color=color, linewidth=3,
            label=f'{epoch} (n={len(epoch_books)})')

ax.set_xlabel('Normalised narrative time (τ)', fontsize=16)
ax.set_ylabel('Structural balance index B(τ)', fontsize=16)
ax.set_title('Balance Index Over Narrative Time by Literary Epoch', fontweight='bold', fontsize=18)
ax.set_xlim(0, 1)
ax.legend(frameon=False, fontsize=11)
plt.tight_layout()
savefig(fig, 'figs_F', 'F1_balance_by_epoch.png')
savefig(fig, 'figs_F', 'F1_balance_by_epoch.pdf')

In [ ]:
# BLOCK F2: Author case studies — 3-panel redesign

# Story: Within a prolific author, genre conventions predict structural balance level, mirroring the corpus-wide pattern.

# Panels:
#   1. Shakespeare (36 works): Tragedy vs Comedy separation — genre means prominent, individual lines very faint
#   2. Jane Austen (6 Romance novels): tight within-genre author consistency
#   3. Margaret Atwood (7 novels, 4 genres): multi-genre author whose works distribute by genre

CASE_STUDY_AUTHORS = ['William Shakespeare', 'Jane Austen', 'Margaret Atwood']

fig, axes = plt.subplots(1, 3, figsize=(16, 6), sharey=True)
axes[0].set_ylabel('B(τ) – Social Balance Index', fontsize=12)

for ax, author in zip(axes, CASE_STUDY_AUTHORS):
    author_books = registry[registry['Author'] == author].copy()
    if len(author_books) == 0:
        ax.set_title(f'{author}\n(not in corpus)')
        continue

    genres_in_author = author_books['Genre'].unique()

    # Step 1: faint individual lines 
    for _, row in author_books.iterrows():
        book  = row['Book']
        genre = row['Genre']
        bdf_b = df_balance[df_balance['Book'] == book].dropna(subset=['B'])
        if len(bdf_b) < 2:
            continue
        interp = np.interp(TAU_BINS, bdf_b.sort_values('tau')['tau'].values, bdf_b.sort_values('tau')['B'].values)
        # Shakespeare: very faint traces to reduce clutter
        alpha_ind  = 0.12 if author == 'William Shakespeare' else 0.45
        lw_ind     = 0.8  if author == 'William Shakespeare' else 1.4
        ax.plot(TAU_BINS, interp,
                color=GENRE_COLORS.get(genre, 'grey'),
                linewidth=lw_ind, alpha=alpha_ind)

    # Step 2: genre-mean overlays (bold)
    legend_handles = []
    for genre in GENRE_ORDER:
        if genre not in genres_in_author:
            continue
        g_books = author_books[author_books['Genre'] == genre]['Book'].tolist()
        traces  = []
        for b in g_books:
            bdf_b = df_balance[df_balance['Book'] == b].dropna(subset=['B'])
            if len(bdf_b) >= 2:
                traces.append(np.interp(TAU_BINS,
                    bdf_b.sort_values('tau')['tau'].values,
                    bdf_b.sort_values('tau')['B'].values))
        if not traces:
            continue
        mean_traj = np.nanmean(traces, axis=0)
        color     = GENRE_COLORS.get(genre, 'grey')
        short_g   = GENRE_LABELS_SHORT.get(genre, genre)
        n_books_g = len(traces)

        lw_mean = 3.0 if author == 'William Shakespeare' else 2.5
        line,   = ax.plot(TAU_BINS, mean_traj,
                          color=color, linewidth=lw_mean,
                          alpha=1.0, zorder=5,
                          label=f'{short_g} (n={n_books_g})')
        legend_handles.append(line)

    # Step 3: axes & annotation
    n_total = len(author_books)
    ax.set_title(f'{author}\n(n={n_total} works)', fontsize=10, fontweight='bold')
    ax.set_xlabel('Normalised narrative time (τ)', fontsize=12)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axhline(0.5, color='grey', linewidth=0.8, linestyle=':', alpha=0.6)
    ax.tick_params(labelsize=8)

    # Legend below each panel
    ax.legend(handles=legend_handles,
              loc='upper center',
              bbox_to_anchor=(0.5, -0.18),
              ncol=2 if len(legend_handles) > 3 else 1,
              frameon=False, fontsize=8)

fig.suptitle(
    'Balance Trajectories for Selected Authors\n'
    'Bold lines = genre means; faint traces = individual works',
    fontsize=18, fontweight='bold', y=1.02
)
plt.subplots_adjust(bottom=0.22, wspace=0.06)
savefig(fig, 'figs_F', 'F2_author_balance_trajectories.png')
savefig(fig, 'figs_F', 'F2_author_balance_trajectories.pdf')
print('F2 saved.')

In [ ]:
# BLOCK F3: Form-stratified Balance B(t) — robustness check

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, form_val in zip(axes, ['Play', 'Novel']):
    df_form_b = df_balance[df_balance['Form'] == form_val]
    results = genre_mean_ci(df_form_b, 'B')
    for genre, data in results.items():
        if data['n_books'] < 3:
            continue
        color = GENRE_COLORS.get(genre, 'grey')
        ax.fill_between(TAU_BINS, data['lower'], data['upper'], color=color, alpha=0.065)
        ax.plot(TAU_BINS, data['mean'], color=color, linewidth=2.3, label=f"{GENRE_LABELS_SHORT.get(genre, genre)} (n={data['n_books']})")
    n = df_form_b['Book'].nunique()
    ax.set_title(f'{form_val}s Only – n={n} books')
    ax.set_xlabel('τ')
    ax.set_ylabel('B(τ)')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

axes[1].legend(frameon=False, fontsize=8, bbox_to_anchor=(1.01,1), loc='upper left')
fig.suptitle('Balance Index B(τ) by Genre – Stratified by Form\n'
             'Robustness check: genre patterns should hold within Plays and Novels separately', fontsize=18, fontweight='bold', y=1.02)
plt.tight_layout()
savefig(fig, 'figs_F', 'F3_balance_form_stratified.png')
savefig(fig, 'figs_F', 'F3_balance_form_stratified.pdf')

---
## Block H — Additional Analyses 
Centrality rank stability and network polarization.
These blocks depend on df_temporal, df_balance, and df (raw) from earlier blocks. Run after Block F.

---
## Block H1 — Centrality Rank Stability
**RQ:** Do early-central characters remain central? Spearman ρ between degree ranks at τ=0.2 and τ=0.8.
High ρ = frozen hierarchy; low ρ = rank reversal (social mobility).
**Genre hypothesis:** Coming-of-Age/Romance (protagonist-heavy) → highest ρ; Mystery (late antagonist reveal) → lowest.

In [ ]:
# BLOCK H1: Centrality Rank Stability ("Social Mobility")

# For each book, compute Spearman rank-correlation of character degree at tau~0.2 (early) vs tau~0.8 (late).
# rho ~ 1.0: frozen social hierarchy (same characters stay central)
# rho << 1 : rank reversal (late-emerging characters gain prominence)

from scipy.stats import spearmanr

rank_stability = []

for book, bdf_b in df.groupby('Book'):
    genre = bdf_b['Genre_Primary'].iloc[0]
    form  = bdf_b['Form'].iloc[0]
    chapters, edge_dicts, graphs = build_cumulative_networks(bdf_b, return_graphs=True)
    if len(chapters) < 4:
        continue
    K = max(chapters)
    taus = [ch / K for ch in chapters]

    # Find snapshots closest to tau = 0.2 (early) and tau = 0.8 (late)
    idx_early = min(range(len(taus)), key=lambda i: abs(taus[i] - 0.20))
    idx_late  = min(range(len(taus)), key=lambda i: abs(taus[i] - 0.80))
    if idx_early == idx_late:
        continue

    G_early = graphs[idx_early]
    G_late  = graphs[idx_late]
    chars_both = set(G_early.nodes()) & set(G_late.nodes())
    if len(chars_both) < 4:
        continue

    chars_list = sorted(chars_both)
    rho, _ = spearmanr(
        [G_early.degree(c) for c in chars_list],
        [G_late.degree(c)  for c in chars_list]
    )
    rank_stability.append({
        'Book': book, 'Genre': genre, 'Form': form,
        'rho': rho, 'n_chars': len(chars_list)
    })

df_rank_stab = pd.DataFrame(rank_stability)
df_rank_stab.to_csv(DIRS['stats'] / 'H1_centrality_rank_stability.csv', index=False)

print(f"Books computed: {len(df_rank_stab)}")
print(f"Overall median rho: {df_rank_stab['rho'].median():.3f}")
print("\nMedian rho by genre (higher = more stable / frozen hierarchy):")
print(df_rank_stab.groupby('Genre')['rho'].median().reindex(GENRE_ORDER).round(3).to_string())

H_rho, p_rho = kruskal(*[df_rank_stab[df_rank_stab['Genre']==g]['rho'].dropna().values
                          for g in GENRE_ORDER if g in df_rank_stab['Genre'].values
                          and len(df_rank_stab[df_rank_stab['Genre']==g]) >= 3])
print(f"\nKruskal-Wallis (rho ~ genre): H={H_rho:.3f}, p={p_rho:.4f}")

# Figure: boxplot of rho by genre
fig, ax = plt.subplots(figsize=(11, 5))
genres_h1 = [g for g in GENRE_ORDER if g in df_rank_stab['Genre'].values]
data_h1   = [df_rank_stab[df_rank_stab['Genre'] == g]['rho'].dropna().values for g in genres_h1]
bp = ax.boxplot(data_h1, patch_artist=True, medianprops=dict(color='black', linewidth=2), flierprops=dict(marker='o', markersize=2, alpha=0.4))
for patch, g in zip(bp['boxes'], genres_h1):
    patch.set_facecolor(GENRE_COLORS[g]); patch.set_alpha(0.85)
ax.set_xticks(range(1, len(genres_h1)+1))
ax.set_xticklabels([GENRE_LABELS_SHORT.get(g, g) for g in genres_h1], fontsize=12)
ax.set_ylabel('Spearman ρ  (degree rank at τ=0.2 vs τ=0.8)')
ax.set_title('Centrality Rank Stability by Genre', fontweight='bold', fontsize=18)
ax.axhline(1.0, color='#aaa', linestyle=':', linewidth=1, label='Perfect stability (ρ=1)')
ax.legend(frameon=False, fontsize=9)
plt.tight_layout()
savefig(fig, 'figs_H', 'H1_centrality_rank_stability.png')
savefig(fig, 'figs_H', 'H1_centrality_rank_stability.pdf')

---
## Block H2 — Network Polarization (Two-Faction Tendency)
**RQ:** Do literary networks show the two-faction structure predicted by Marvel et al. (2011)?  
Proxy: fraction of negative edges that fall *between* (not within) Louvain communities.  
High = two hostile groups; low = internal conflict.

In [ ]:
# BLOCK H2: Network Polarization — Two-Faction Tendency

# Tests Marvel et al. (2011): signed networks tend toward a "two hostile camps" configuration under balance dynamics.
# Proxy: fraction of negative edges that fall BETWEEN (not within) the network's communities (Louvain on unsigned graph).
# High between-fraction → two-faction / polarized structure. Low between-fraction → conflict is internal, not inter-group.

from networkx.algorithms.community import greedy_modularity_communities

faction_results = []

for book, bdf_b in df.groupby('Book'):
    genre = bdf_b['Genre_Primary'].iloc[0]
    form  = bdf_b['Form'].iloc[0]
    _, _, graphs = build_cumulative_networks(bdf_b, return_graphs=True)
    G_final = graphs[-1]
    if G_final.number_of_nodes() < 6 or G_final.number_of_edges() < 5:
        continue

    try:
        communities = list(greedy_modularity_communities(G_final))
    except Exception:
        continue
    if len(communities) < 2:
        continue

    node_comm = {node: cid for cid, comm in enumerate(communities) for node in comm}

    neg_between, total_neg = 0, 0
    for u, v, d in G_final.edges(data=True):
        if d.get('polarity') == 'negative':
            total_neg += 1
            if node_comm.get(u, -1) != node_comm.get(v, -2):
                neg_between += 1

    if total_neg == 0:
        continue
    faction_results.append({
        'Book': book, 'Genre': genre, 'Form': form,
        'neg_between_frac': neg_between / total_neg,
        'n_communities': len(communities),
    })

df_faction = pd.DataFrame(faction_results)
df_faction.to_csv(DIRS['stats'] / 'H2_polarization.csv', index=False)

print(f"Books computed: {len(df_faction)}")
print("\nMean between-community negative fraction by genre:")
print(df_faction.groupby('Genre')['neg_between_frac'].mean().reindex(GENRE_ORDER).round(3).to_string())
print("\n(>0.5 = more conflict between communities than within → two-faction)")

fig, ax = plt.subplots(figsize=(11, 5))
genres_h2 = [g for g in GENRE_ORDER if g in df_faction['Genre'].values]
data_h2   = [df_faction[df_faction['Genre']==g]['neg_between_frac'].dropna().values for g in genres_h2]
bp = ax.boxplot(data_h2, patch_artist=True, medianprops=dict(color='black', linewidth=2), flierprops=dict(marker='o', markersize=2, alpha=0.4))
for patch, g in zip(bp['boxes'], genres_h2):
    patch.set_facecolor(GENRE_COLORS[g]); patch.set_alpha(0.85)
ax.axhline(0.5, color='#d62728', linestyle='--', linewidth=1.3, label='Random baseline (0.5)')
ax.set_xticks(range(1, len(genres_h2)+1))
ax.set_xticklabels([GENRE_LABELS_SHORT.get(g, g) for g in genres_h2], fontsize=10)
ax.set_ylabel('Fraction of negative edges between communities')
ax.set_title('Network Polarization by Genre', fontsize=18, fontweight='bold')
ax.legend(frameon=False, fontsize=9)
plt.tight_layout()
savefig(fig, 'figs_H', 'H2_polarization.png')
savefig(fig, 'figs_H', 'H2_polarization.pdf')

---
## Block G — Per-Book Outputs
Saves individual analysis files for each book to `per_book/{Author}_{Title}/`

In [ ]:
# BLOCK G_DEFS: Per-book helper function definitions

# Defines: sanitize_dirname(), save_per_book_outputs(), _save_per_book_figure(), _save_polarity_heatmap()

# Run ONCE before G0 (test) and G1 (full loop).
# Outputs per book:
#   per_book/<Title>/
#     ├── network_stats.json
#     ├── temporal_metrics.csv
#     ├── per_book_story.png       (9-panel analytical figure)
#     └── polarity_heatmap.png    (standalone named-pair heatmap)

# ─── Consistent polarity palette ───────────────────────────
_POL_COL = {'positive': '#2ca02c', 'negative': '#d62728', 'neutral': '#aec7e8'}
from matplotlib.colors import ListedColormap, BoundaryNorm
_DISC_CMAP = ListedColormap(['#d62728', '#aec7e8', '#2ca02c'])
_DISC_NORM = BoundaryNorm([-1.5, -0.5, 0.5, 1.5], _DISC_CMAP.N)
import matplotlib.patches as _mp

# Load Year lookup from metadata
try:
    _tmp = pd.read_csv(METADATA_CSV)
    _tmp['_key'] = (_tmp['Author'] + ': ' + _tmp['Book']).replace(BOOK_KEY_FIXES)
    _meta_year = _tmp.drop_duplicates('_key').set_index('_key')['Year'].to_dict()
except Exception:
    _meta_year = {}

def sanitize_dirname(s):
    return s.replace('/', '_').replace(':', '_').replace(' ', '_')[:80]

# Standalone polarity heatmap with full named character pairs on y-axis
def _save_polarity_heatmap(book, bdf, book_dir):
    chapters = sorted(bdf['Chapter'].unique())
    pair_freq = bdf['pair'].value_counts()
    top_pairs = pair_freq.head(35).index.tolist()
    if len(top_pairs) == 0:
        return

    pol_map = {'positive': 1, 'neutral': 0, 'negative': -1}
    matrix  = np.full((len(top_pairs), len(chapters)), np.nan)
    for j, ch in enumerate(chapters):
        for row in bdf[bdf['Chapter'] == ch].itertuples():
            if row.pair in top_pairs:
                matrix[top_pairs.index(row.pair), j] = pol_map.get(row.Relationship, np.nan)

    pair_labels = [f'{a[:20]}–{b[:20]}' for a, b in top_pairs]
    fig_h = max(8, len(top_pairs) * 0.38)
    fig_w = max(10, len(chapters) * 0.50)
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))

    ax.imshow(matrix, aspect='auto', cmap=_DISC_CMAP, norm=_DISC_NORM, interpolation='nearest')
    ax.set_xticks(range(len(chapters)))
    ax.set_xticklabels([str(c) for c in chapters], rotation=90, fontsize=7.5)
    ax.set_yticks(range(len(top_pairs)))
    ax.set_yticklabels(pair_labels, fontsize=7.5)
    ax.set_xlabel('Chapter', fontsize=10)
    ax.set_ylabel('Character pair', fontsize=10)
    year = _meta_year.get(book, '')
    ax.set_title(f'{book}  [{year}]\nCharacter Relationship Polarity Heatmap', fontsize=11)

    leg = [_mp.Patch(color='#2ca02c', label='Positive'),
           _mp.Patch(color='#aec7e8', label='Neutral'),
           _mp.Patch(color='#d62728', label='Negative')]
    ax.legend(handles=leg, loc='upper right', bbox_to_anchor=(1.15, 1.0),
              frameon=True, fontsize=9)

    plt.tight_layout()
    fig.savefig(book_dir / 'polarity_heatmap.png', dpi=150, bbox_inches='tight')
    plt.close(fig)

# Save per-book outputs: JSON stats, temporal CSV, 9-panel figure, standalone heatmap
def save_per_book_outputs(book, bdf, static_row, temp_df_b, bal_df_b):
    book_dir = DIRS['per_book'] / sanitize_dirname(book)
    book_dir.mkdir(parents=True, exist_ok=True)

    # 1. Temporal metrics CSV
    temp_df_b.to_csv(book_dir / 'temporal_metrics.csv', index=False)

    # 2. Network stats JSON
    if static_row is not None:
        stats_dict = {k: (None if (isinstance(v, float) and np.isnan(v)) else v)
                      for k, v in static_row.items()}
        with open(book_dir / 'network_stats.json', 'w') as fj:
            json.dump(stats_dict, fj, indent=2)

    # 3. Standalone polarity heatmap
    try:
        _save_polarity_heatmap(book, bdf, book_dir)
    except Exception as e:
        log.warning(f'Polarity heatmap failed for {book}: {e}')

    # 4. Nine-panel analytical figure
    try:
        _save_per_book_figure(book, bdf, static_row, temp_df_b, bal_df_b, book_dir)
    except Exception as e:
        log.warning(f'Per-book figure failed for {book}: {e}')

# Show a clean placeholder with explanation when a panel cannot be drawn
def _empty_panel(ax, reason):
    ax.axis('off')
    ax.text(0.5, 0.5, reason, ha='center', va='center', transform=ax.transAxes,
            fontsize=8, color='#888888', style='italic',
            wrap=True, multialignment='center')

# Nine-panel per-book analytical figure
def _save_per_book_figure(book, bdf, static_row, temp_df_b, bal_df_b, book_dir):
    genre = bdf['Genre_Primary'].iloc[0]
    form  = bdf['Form'].iloc[0]
    epoch = bdf['Epoch'].iloc[0] if 'Epoch' in bdf.columns else ''
    year  = _meta_year.get(book, '')

    chapters_sorted = sorted(bdf['Chapter'].unique())
    K    = max(chapters_sorted)
    taus = [ch / K for ch in chapters_sorted]

    tmp = temp_df_b.sort_values('tau').dropna(subset=['tau'])
    bal = bal_df_b.sort_values('tau').dropna(subset=['B']) if (bal_df_b is not None and len(bal_df_b) > 0) else pd.DataFrame()

    fig = plt.figure(figsize=(18, 12))
    year_str = f'  [{year}]' if year else ''
    fig.suptitle(f'{book}{year_str}\n{genre}  |  {form}  |  {epoch}',
                 fontsize=12, fontweight='bold', y=0.99)
    gs = fig.add_gridspec(3, 3, hspace=0.46, wspace=0.38)

    # Panel 1: Cast & Edge Growth
    ax1 = fig.add_subplot(gs[0, 0])
    # Correct column names: N_t and E_t (from Block B1)
    has_N = 'N_t' in tmp.columns and tmp['N_t'].notna().any()
    has_E = 'E_t' in tmp.columns and tmp['E_t'].notna().any()
    if has_N or has_E:
        if has_N:
            ax1.plot(tmp['tau'], tmp['N_t'], color='#1f77b4', lw=2, label='Nodes N')
        if has_E:
            ax1r = ax1.twinx()
            ax1r.plot(tmp['tau'], tmp['E_t'], color='#ff7f0e', lw=2, ls='--', label='Edges E')
            ax1r.set_ylabel('Edges E', fontsize=8, color='#ff7f0e')
            ax1r.tick_params(axis='y', labelcolor='#ff7f0e', labelsize=7)
        ax1.set_xlabel(r'$\tau$', fontsize=8)
        ax1.set_ylabel('Characters N', fontsize=8, color='#1f77b4')
        ax1.tick_params(labelsize=7)
    else:
        _empty_panel(ax1, 'Cast/edge data\nnot available for\nthis book')
    ax1.set_title('Cast & Edge Growth', fontsize=9)

    # Panel 2: Structural Balance B(τ)
    ax2 = fig.add_subplot(gs[0, 1])
    if len(bal) >= 3:
        ax2.plot(bal['tau'], bal['B'], color='#6f1926', lw=2.5, label=r'$B(\tau)$ strong')
        if 'B_weak' in bal.columns and bal['B_weak'].notna().any():
            ax2.plot(bal['tau'], bal['B_weak'], color='#e07070', lw=1.5,
                     ls='--', alpha=0.7, label=r'$B_w(\tau)$ weak')
        ax2.axhline(0.5, color='grey', lw=0.8, ls=':', label='random (0.5)')
        ax2.legend(fontsize=6.5, frameon=False)
        ax2.set_xlim(0, 1); ax2.set_ylim(0, 1)
        ax2.set_xlabel(r'$\tau$', fontsize=8)
        ax2.set_ylabel(r'$B(\tau)$', fontsize=8)
        ax2.tick_params(labelsize=7)
    else:
        _empty_panel(ax2, 'Balance not computed\n(< 3 signed triads\nin this book)')
    ax2.set_title('Structural Balance', fontsize=9)

    # Panel 3: Signed Density D+/D- 
    ax3 = fig.add_subplot(gs[0, 2])
    has_dpos = 'd_pos' in tmp.columns and tmp['d_pos'].notna().any()
    has_dneg = 'd_neg' in tmp.columns and tmp['d_neg'].notna().any()
    if has_dpos or has_dneg:
        if has_dpos:
            ax3.plot(tmp['tau'], tmp['d_pos'], color=_POL_COL['positive'], lw=2, label=r'$D^+$')
        if has_dneg:
            ax3.plot(tmp['tau'], tmp['d_neg'], color=_POL_COL['negative'], lw=2, label=r'$D^-$')
        ax3.set_xlabel(r'$\tau$', fontsize=8); ax3.set_ylabel('Signed density', fontsize=8)
        ax3.legend(fontsize=7, frameon=False); ax3.tick_params(labelsize=7)
    else:
        _empty_panel(ax3, 'Signed density\nnot available')
    ax3.set_title('Positive vs Negative Density', fontsize=9)

    # Panel 4: Alliance Ratio R+(τ)
    ax4 = fig.add_subplot(gs[1, 0])
    if 'r_plus' in tmp.columns and tmp['r_plus'].notna().any():
        rp = tmp['r_plus'].values
        ax4.plot(tmp['tau'], rp, color='#2ca02c', lw=2)
        ax4.axhline(0.5, color='grey', lw=0.8, ls=':')
        ax4.fill_between(tmp['tau'], 0.5, rp, where=(rp >= 0.5), alpha=0.15, color='#2ca02c')
        ax4.fill_between(tmp['tau'], 0.5, rp, where=(rp  < 0.5), alpha=0.15, color='#d62728')
        ax4.set_xlim(0, 1); ax4.set_ylim(0, 1)
        ax4.set_xlabel(r'$\tau$', fontsize=8); ax4.set_ylabel(r'$R^+$', fontsize=8)
        ax4.tick_params(labelsize=7)
    else:
        _empty_panel(ax4, 'Alliance ratio\nnot available\n(no +/− edges)')
    ax4.set_title('Alliance Ratio', fontsize=9)

    # Panel 5: Degree Entropy H(τ) 
    ax5 = fig.add_subplot(gs[1, 1])
    if 'H_deg' in tmp.columns and tmp['H_deg'].notna().any():
        ax5.plot(tmp['tau'], tmp['H_deg'], color='#9467bd', lw=2)
        ax5.set_xlabel(r'$\tau$', fontsize=8); ax5.set_ylabel(r'$H(\tau)$', fontsize=8)
        ax5.tick_params(labelsize=7)
    else:
        _empty_panel(ax5, 'Entropy not available')
    ax5.set_title('Social Egalitarianism (Entropy)', fontsize=9)

    # Panel 6: Polarity heatmap (pairs × chapters)
    ax6 = fig.add_subplot(gs[1, 2])
    bdf_s     = bdf.sort_values('Chapter')
    all_pairs = bdf_s['pair'].value_counts().head(20).index.tolist()
    chapters_h = sorted(bdf_s['Chapter'].unique())
    pol_map = {'positive': 1, 'neutral': 0, 'negative': -1}
    mat = np.full((len(all_pairs), len(chapters_h)), np.nan)
    for jc, ch in enumerate(chapters_h):
        for row in bdf_s[bdf_s['Chapter'] == ch].itertuples():
            if row.pair in all_pairs:
                mat[all_pairs.index(row.pair), jc] = pol_map.get(row.Relationship, np.nan)
    if len(all_pairs) > 0:
        ax6.imshow(mat, aspect='auto', cmap=_DISC_CMAP, norm=_DISC_NORM, interpolation='nearest')
        ax6.set_xlabel('Chapter', fontsize=8); ax6.set_ylabel('Pair', fontsize=8)
        ax6.set_yticks([]); ax6.tick_params(labelsize=6)
    else:
        _empty_panel(ax6, 'No character pairs found')
    ax6.set_title('Polarity Heatmap (top-20 pairs)', fontsize=9)

    # Panel 7: Character Introduction Curve
    ax7 = fig.add_subplot(gs[2, 0])
    chars_seen = set(); cumulative = []
    for ch in chapters_sorted:
        ch_chars = set(bdf[bdf['Chapter'] == ch]['Character A'].tolist() + bdf[bdf['Chapter'] == ch]['Character B'].tolist())
        chars_seen |= ch_chars
        cumulative.append(len(chars_seen))
    if cumulative:
        ax7.plot(taus, cumulative, color='#1f77b4', lw=2)
        ax7.set_xlabel(r'$\tau$', fontsize=8); ax7.set_ylabel('Cumulative chars', fontsize=8)
        ax7.tick_params(labelsize=7)
    else:
        _empty_panel(ax7, 'No character data')
    ax7.set_title('Cast Introduction Curve', fontsize=9)

    # Panel 8: Top-5 Character Centrality
    ax8 = fig.add_subplot(gs[2, 1])
    try:
        _, _, graphs_pb = build_cumulative_networks(bdf, return_graphs=True)
        chapters_pb = sorted(bdf['Chapter'].unique()); K_pb = max(chapters_pb)
        G_final = graphs_pb[-1]
        top5 = sorted(G_final.nodes(), key=lambda n: G_final.degree(n), reverse=True)[:5]
        cmap5 = plt.cm.tab10.colors
        for ci, char in enumerate(top5):
            degs = []; tpb = []
            for ch, G in zip(chapters_pb, graphs_pb):
                if char in G:
                    degs.append(G.degree(char)); tpb.append(ch / K_pb)
            ax8.plot(tpb, degs, lw=1.8, color=cmap5[ci], label=str(char)[:14])
        ax8.legend(fontsize=5.5, frameon=False, loc='upper left')
        ax8.set_xlabel(r'$\tau$', fontsize=8); ax8.set_ylabel('Degree', fontsize=8)
        ax8.tick_params(labelsize=7)
    except Exception:
        _empty_panel(ax8, 'Centrality not\navailable')
    ax8.set_title('Top-5 Character Centrality', fontsize=9)

    # Panel 9: Static Summary
    ax9 = fig.add_subplot(gs[2, 2])
    ax9.axis('off')
    if static_row is not None:
        def _fmt(v, fmt='.3f'):
            return format(v, fmt) if isinstance(v, float) and not np.isnan(v) else '—'
        lines_txt = [
            f"N = {static_row.get('N', '—')}  |  E = {static_row.get('E', '—')}",
            f"Density = {_fmt(static_row.get('density', float('nan')))}",
            f"σ_sw    = {_fmt(static_row.get('sigma_sw', float('nan')), '.2f')}",
            f"C_avg   = {_fmt(static_row.get('C_avg',   float('nan')), '.2f')}",
            f"f+ = {_fmt(static_row.get('f_pos', float('nan')), '.2f')}  "
            f"f0 = {_fmt(static_row.get('f_neu', float('nan')), '.2f')}  "
            f"f- = {_fmt(static_row.get('f_neg', float('nan')), '.2f')}",
            f"\nGenre:  {genre}",
            f"Form:   {form}",
            f"Year:   {year if year else '—'}",
            f"Epoch:  {epoch}",
        ]
        ax9.text(0.05, 0.95, '\n'.join(lines_txt), transform=ax9.transAxes, fontsize=8.5, va='top', family='monospace', bbox=dict(boxstyle='round,pad=0.4', fc='#f8f8f8', ec='#cccccc'))
    ax9.set_title('Network Summary', fontsize=9)

    fig.savefig(book_dir / 'per_book_story.png', dpi=150, bbox_inches='tight')
    plt.close(fig)

print('G_DEFS ready: sanitize_dirname, save_per_book_outputs, _save_per_book_figure, _save_polarity_heatmap')
print('Run G0 (test 3 books) → then G1 (all books).')


In [ ]:
# BLOCK G0: Per-book test run — 3 sample books

# Run this cell FIRST to verify per-book output looks correct
# before triggering the full 600+ book loop in Block G1.
# Inspect the output directories for these 3 books, then proceed.

TEST_BOOKS = books_list[:10]  # change indices to inspect specific books

print(f'Test run on {len(TEST_BOOKS)} books:')
for book in TEST_BOOKS:
    print(f'  - {book}')
    bdf = df[df['Book'] == book]

    # Build static row
    static_row_test = df_static[df_static['Book'] == book].iloc[0].to_dict() if book in df_static['Book'].values else None

    # Build temporal + balance slices
    temp_df_test  = df_temporal[df_temporal['Book'] == book]
    bal_df_test   = df_balance[df_balance['Book'] == book]

    save_per_book_outputs(book, bdf, static_row_test, temp_df_test, bal_df_test)
    print(f'    → Saved to: {DIRS["per_book"] / sanitize_dirname(book)}')

print('\n Test complete. Inspect the above directories, then run Block G1 for all books.')

In [ ]:
# BLOCK G1 (cont.): Run per-book output loop for all books

# Run Block G0 (test cell above) first to verify outputs for 3 books, then run this cell to process all books. Takes ~10-30 min.
# sanitize_dirname and save_per_book_outputs are defined in the cell above.

log.info('Saving per-book outputs...')
t0 = time.time()

# Index lookup structures
static_index  = df_static.set_index('Book')
temp_index    = df_temporal.groupby('Book')
balance_index = df_balance.groupby('Book')

for i, book in enumerate(books_list):
    bdf = df[df['Book'] == book]
    static_row = static_index.loc[book].to_dict() if book in static_index.index else None
    temp_df_b  = temp_index.get_group(book) if book in temp_index.groups else pd.DataFrame()
    bal_df_b   = balance_index.get_group(book) if book in balance_index.groups else pd.DataFrame()
    save_per_book_outputs(book, bdf, static_row, temp_df_b, bal_df_b)

    if (i + 1) % 100 == 0:
        log.info(f'  Per-book outputs: {i+1}/{len(books_list)} ({time.time()-t0:.0f}s)')

print(f' Block G1 complete in {time.time()-t0:.0f}s')
print(f'   Per-book folders saved to: {DIRS["per_book"]}')

In [ ]:
# BLOCK G2: Featured polarity heatmap

# Discrete 3-colour scheme (no gradient): green=positive, blue=neutral, red=negative

FEATURED_BOOK = "Margaret Atwood: Alias Grace"
# Other options: "Agatha Christie: The Murder of Roger Ackroyd",
#                "George Orwell: 1984", "F Scott Fitzgerald: The Great Gatsby"

if FEATURED_BOOK not in df['Book'].values:
    candidates = [b for b in df['Book'].unique() if 'Atwood' in b]
    FEATURED_BOOK = candidates[0] if candidates else df['Book'].unique()[50]
    print(f'Using: {FEATURED_BOOK}')

bdf_feat = df[df['Book'] == FEATURED_BOOK]
chapters  = sorted(bdf_feat['Chapter'].unique())
pair_freq = bdf_feat['pair'].value_counts()
top_pairs = pair_freq.head(30).index.tolist()

# Discrete colour map: negative=-1 → red, neutral=0 → light blue, positive=1 → green
from matplotlib.colors import ListedColormap, BoundaryNorm
disc_cmap  = ListedColormap(['#d62728', '#aec7e8', '#2ca02c'])  # red, neutral-blue, green
disc_norm  = BoundaryNorm([-1.5, -0.5, 0.5, 1.5], disc_cmap.N)

# Build matrix
pol_map = {'positive': 1, 'neutral': 0, 'negative': -1}
matrix  = np.full((len(top_pairs), len(chapters)), np.nan)
for j, ch in enumerate(chapters):
    for row in bdf_feat[bdf_feat['Chapter'] == ch].itertuples():
        if row.pair in top_pairs:
            matrix[top_pairs.index(row.pair), j] = pol_map.get(row.Relationship, np.nan)

pair_labels = [f'{a[:18]}–{b[:18]}' for a, b in top_pairs]

fig, ax = plt.subplots(figsize=(max(10, len(chapters) * 0.5), max(8, len(top_pairs) * 0.38)))
ax.imshow(matrix, aspect='auto', cmap=disc_cmap, norm=disc_norm, interpolation='nearest')

ax.set_xticks(range(len(chapters)))
ax.set_xticklabels([str(c) for c in chapters], rotation=90, fontsize=7.5)
ax.set_yticks(range(len(top_pairs)))
ax.set_yticklabels(pair_labels, fontsize=8)
ax.set_xlabel('Chapter', fontsize=11)
ax.set_ylabel('Character pair', fontsize=11)
ax.set_title(f'{FEATURED_BOOK}\nCharacter Relationship Polarity Heatmap', fontsize=12)

# Discrete legend patches instead of colorbar
import matplotlib.patches as mpatches
leg = [mpatches.Patch(color='#2ca02c', label='Positive'),
       mpatches.Patch(color='#aec7e8', label='Neutral'),
       mpatches.Patch(color='#d62728', label='Negative')]
ax.legend(handles=leg, loc='upper right', bbox_to_anchor=(1.18, 1.0),
          frameon=True, fontsize=9)

plt.tight_layout()
savefig(fig, 'figs_A', 'G2_featured_polarity_heatmap.png')